In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║           SMARTINVEST 4.4 — AGENTIC PROTOTYPE EDITION                          ║
║        Autonomous AI Agents + AI Infrastructure Diversification                 ║
║                                                                                  ║
║  What's new vs 4.3:                                                              ║
║  • AgenticPrototype — autonomous agents for anomaly detection & portfolio adj.  ║
║  • AIReadyDataModule — AI-optimized data preparation + FinBERT sentiment        ║
║  • DiversificationModule — AI infra assets (cybersecurity, data centers, optics)║
║  • IntegrityLayer — model poisoning detection + decision trace audit            ║
║  • ROITracker — real AI investment ROI + bubble warning system                  ║
║  • ChiefAIOfficerMode — enterprise AI leadership simulation                     ║
║  • Extended Plotly dashboard with AI Factories panel                            ║
║  • All 4.3 features retained (adaptive weights, TFT, FRED, Black-Litterman…)   ║
╚══════════════════════════════════════════════════════════════════════════════════╝

VERSION: 4.4.0
CODENAME: Agentic Prototype

DISCLAIMER: Educational software. NOT financial advice.
You CAN and MAY LOSE MONEY. Consult a licensed financial advisor.
Past performance does NOT guarantee future results.
"""

# =============================================================================
# IMPORTS
# =============================================================================

import os
import sys
import json
import time
import sqlite3
import logging
import pickle
import warnings
import asyncio
import hashlib
import traceback
import threading
from pathlib import Path
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Optional, Tuple, Any, Callable
from enum import Enum
from concurrent.futures import ThreadPoolExecutor, as_completed
from logging.handlers import RotatingFileHandler

import numpy as np
import pandas as pd
from scipy.optimize import minimize, linprog
from scipy import stats
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.isotonic import IsotonicRegression
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.covariance import EllipticEnvelope

import yfinance as yf

warnings.filterwarnings('ignore')

# --- Optional dependencies ---
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import (
        Dense, LSTM, Dropout, Input,
        MultiHeadAttention, LayerNormalization,
        Add, Multiply, Lambda
    )
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    TF_AVAILABLE = True
except ImportError:
    TF_AVAILABLE = False

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
except ImportError:
    TEXTBLOB_AVAILABLE = False

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

try:
    from transformers import pipeline as hf_pipeline
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

try:
    import networkx as nx
    NETWORKX_AVAILABLE = True
except ImportError:
    NETWORKX_AVAILABLE = False

try:
    import pulp
    PULP_AVAILABLE = True
except ImportError:
    PULP_AVAILABLE = False


# =============================================================================
# MODULE 1: SECURE CONFIGURATION (Extended for 4.4)
# =============================================================================

class SecureConfig:
    """
    Centralized, immutable configuration.
    All API keys loaded from environment variables — never hardcoded.
    Extended in 4.4 with agentic, AI-ready data, and AI infra settings.
    """

    # --- Paths ---
    BASE_DIR: Path = Path.cwd()
    CACHE_DIR: Path = BASE_DIR / "si44_cache"
    MODELS_DIR: Path = BASE_DIR / "si44_models"
    LOGS_DIR: Path = BASE_DIR / "si44_logs"
    DECISIONS_DIR: Path = BASE_DIR / "si44_decisions"
    DB_PATH: Path = CACHE_DIR / "smartinvest44.db"
    TRACES_DIR: Path = BASE_DIR / "si44_traces"

    # --- API keys (env vars only) ---
    FINNHUB_API_KEY: str = os.getenv("FINNHUB_API_KEY", "")
    FRED_API_KEY: str = os.getenv("FRED_API_KEY", "")
    NEWSAPI_KEY: str = os.getenv("NEWSAPI_KEY", "")
    POLYGON_API_KEY: str = os.getenv("POLYGON_API_KEY", "")

    # --- Risk parameters ---
    MAX_POSITION_SIZE: float = 0.20
    MIN_POSITION_SIZE: float = 0.02
    MAX_DRAWDOWN_LIMIT: float = -0.15
    CIRCUIT_BREAKER_5D_LOSS: float = -0.10
    CRISIS_VIX_THRESHOLD: float = 28.0
    CRISIS_MAX_POSITION: float = 0.05

    # --- Model parameters ---
    LOOKBACK_WINDOW: int = 60
    FORECAST_HORIZON: int = 30
    MIN_HISTORY_DAYS: int = 504
    TFT_EPOCHS: int = 30
    LSTM_EPOCHS: int = 20
    BATCH_SIZE: int = 32

    # --- Ensemble weights (initial) ---
    INITIAL_ENSEMBLE_WEIGHTS: Dict[str, float] = {
        'tft':   0.30,
        'lstm':  0.25,
        'xgb':   0.20,
        'lgbm':  0.15,
        'rf':    0.10,
    }
    MIN_MODEL_WEIGHT: float = 0.05
    WEIGHT_UPDATE_INTERVAL: int = 50
    WEIGHT_TEMPERATURE: float = 2.0

    # --- Decision engine thresholds ---
    MIN_CONFIDENCE: float = 0.45
    MAX_MODEL_DISAGREEMENT: float = 0.025
    MAX_STRESS_INDEX: float = 0.70
    MIN_REGIME_HIT_RATE: float = 0.50
    MAX_SECTOR_CONCENTRATION: float = 0.40
    MIN_VOLUME_RATIO: float = 0.30

    # --- Calibration ---
    MIN_CALIBRATION_SAMPLES: int = 30
    CALIBRATION_UPDATE_DAYS: int = 7

    # --- Opportunity cost values ---
    OUTCOME_VALUES: Dict[str, float] = {
        'correct_trade':      3.0,
        'wrong_trade':       -6.0,
        'missed_opportunity': -1.0,
        'neutral_hold':       0.0,
    }

    # --- Data cache TTL ---
    MARKET_CACHE_TTL: int = 4 * 3600
    MACRO_CACHE_TTL: int = 24 * 3600

    # =====================================================================
    # 4.4 NEW: Agentic + AI-Ready + AI Infra settings
    # =====================================================================

    # Agentic settings
    AGENTIC_ENABLED: bool = True
    AGENT_ANOMALY_THRESHOLD: float = 2.5       # Z-score for anomaly detection
    AGENT_REBALANCE_TRIGGER: float = 0.05      # 5% drift triggers rebalance
    AGENT_MAX_AUTO_ADJUST: float = 0.10        # Max 10% auto position adjustment
    AGENT_TASK_TIMEOUT: int = 30               # Seconds per agent task

    # AI-Ready Data settings
    AI_READY_DATA_THRESHOLD: float = 0.85     # Min completeness ratio
    FINBERT_MODEL: str = "ProsusAI/finbert"    # FinBERT model for sentiment
    DSLM_ENABLED: bool = True                  # Domain-Specific Language Models

    # AI Infrastructure categories
    AI_INFRA_CATEGORIES: List[str] = [
        'cybersecurity', 'data_centers', 'optics', 'robotics', 'rare_earth'
    ]

    # AI Infra instruments per category
    AI_INFRA_INSTRUMENTS: Dict[str, List[str]] = {
        'cybersecurity': ['CRWD', 'PANW', 'ZS', 'FTNT'],        # CrowdStrike, Palo Alto, Zscaler, Fortinet
        'data_centers':  ['EQIX', 'DLR', 'AMT', 'IREN'],        # Equinix, Digital Realty, American Tower, Iris Energy
        'optics':        ['AMAT', 'LRCX', 'ONTO', 'IIVI'],       # Applied Materials, Lam Research, Onto Innovation, II-VI
        'robotics':      ['ABB', 'ROK', 'FANUY', 'IRBT'],        # ABB, Rockwell, Fanuc, iRobot
        'rare_earth':    ['MP', 'UUUU', 'NXE', 'LTHM'],          # MP Materials, Energy Fuels, NexGen, Livent
    }

    # Max exposure to AI infra as % of portfolio
    MAX_AI_INFRA_EXPOSURE: float = 0.35

    # ROI tracker settings
    AI_BUBBLE_WARNING_THRESHOLD: float = 3.0  # P/E ratio multiplier vs historical
    ROI_BENCHMARK_SYMBOL: str = 'SPY'

    # Integrity layer
    POISONING_DETECTION_ENABLED: bool = True
    MODEL_DRIFT_THRESHOLD: float = 0.20        # 20% prediction drift = alert

    @classmethod
    def setup(cls) -> None:
        for d in [cls.CACHE_DIR, cls.MODELS_DIR, cls.LOGS_DIR,
                  cls.DECISIONS_DIR, cls.TRACES_DIR]:
            d.mkdir(parents=True, exist_ok=True)

    @classmethod
    def validate(cls) -> List[str]:
        warnings_list = []
        if not cls.FRED_API_KEY:
            warnings_list.append("FRED_API_KEY not set — macro data limited (free at fred.stlouisfed.org)")
        if not cls.FINNHUB_API_KEY:
            warnings_list.append("FINNHUB_API_KEY not set — news sentiment unavailable")
        if not cls.POLYGON_API_KEY:
            warnings_list.append("POLYGON_API_KEY not set — Polygon finance data unavailable")
        if not TRANSFORMERS_AVAILABLE:
            warnings_list.append("transformers not installed — FinBERT sentiment unavailable (pip install transformers)")
        if not NETWORKX_AVAILABLE:
            warnings_list.append("networkx not installed — integrity graph analysis limited (pip install networkx)")
        if not PULP_AVAILABLE:
            warnings_list.append("pulp not installed — LP optimization fallback active (pip install pulp)")
        return warnings_list

    @classmethod
    def validate_ai_infra_access(cls) -> Dict[str, bool]:
        """Check which AI infra categories have valid API access."""
        access = {}
        for cat in cls.AI_INFRA_CATEGORIES:
            access[cat] = len(cls.AI_INFRA_INSTRUMENTS.get(cat, [])) > 0
        return access


SecureConfig.setup()


# =============================================================================
# MODULE 2: PRODUCTION LOGGER (unchanged from 4.3)
# =============================================================================

class LogLevel(Enum):
    INFO = "INFO"
    WARNING = "WARNING"
    ERROR = "ERROR"
    CRITICAL = "CRITICAL"
    SUCCESS = "SUCCESS"
    PERF = "PERF"
    AGENT = "AGENT"
    INTEGRITY = "INTEGRITY"
    ROI = "ROI"


class ProductionLogger:
    ICONS = {
        LogLevel.INFO:      "ℹ️ ",
        LogLevel.WARNING:   "⚠️ ",
        LogLevel.ERROR:     "❌",
        LogLevel.CRITICAL:  "💀",
        LogLevel.SUCCESS:   "✅",
        LogLevel.PERF:      "⏱️ ",
        LogLevel.AGENT:     "🤖",
        LogLevel.INTEGRITY: "🛡️ ",
        LogLevel.ROI:       "💰",
    }

    def __init__(self):
        self._logger = logging.getLogger("SmartInvest44")
        self._logger.setLevel(logging.DEBUG)
        if not self._logger.handlers:
            self._setup_handlers()
        self._timers: Dict[str, float] = {}

    def _setup_handlers(self):
        fmt = logging.Formatter('%(asctime)s %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
        fh = RotatingFileHandler(
            SecureConfig.LOGS_DIR / "smartinvest44.log",
            maxBytes=10 * 1024 * 1024,
            backupCount=5,
            encoding='utf-8'
        )
        fh.setLevel(logging.DEBUG)
        fh.setFormatter(fmt)
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        ch.setFormatter(fmt)
        self._logger.addHandler(fh)
        self._logger.addHandler(ch)

    def log(self, level: LogLevel, message: str, component: str = "SYSTEM",
            context: Optional[Dict] = None) -> None:
        icon = self.ICONS.get(level, "  ")
        entry = {
            "ts": datetime.now().isoformat(),
            "level": level.value,
            "component": component,
            "msg": message,
        }
        if context:
            entry["ctx"] = context
        log_line = f"{icon} [{component}] {message}"
        if level in (LogLevel.ERROR, LogLevel.CRITICAL):
            self._logger.error(log_line)
        elif level == LogLevel.WARNING:
            self._logger.warning(log_line)
        else:
            self._logger.info(log_line)

    def start_timer(self, name: str) -> None:
        self._timers[name] = time.perf_counter()

    def end_timer(self, name: str) -> float:
        elapsed = time.perf_counter() - self._timers.pop(name, time.perf_counter())
        self.log(LogLevel.PERF, f"{name} completed in {elapsed:.2f}s", "TIMER")
        return elapsed


logger = ProductionLogger()


# =============================================================================
# MODULE 3: ROBUST DATA MANAGER (Extended with AI-Ready Data)
# =============================================================================

class RobustDataManager:
    """
    Multi-source data fetching with SQLite caching, data quality scoring,
    retry logic, and AI-ready data pipeline (4.4 extension).
    """

    FRED_SERIES = {
        'CPIAUCSL':  'CPI',
        'GDP':       'GDP',
        'UNRATE':    'Unemployment',
        'FEDFUNDS':  'FedFunds',
        'T10Y2Y':    'YieldSpread',
        'VIXCLS':    'VIX_FRED',
        'UMCSENT':   'ConsumerSent',
        'INDPRO':    'IndustrialProd',
    }

    # Sector mapping (extended with AI infra sectors in 4.4)
    SECTOR_MAP = {
        'AAPL': 'tech', 'MSFT': 'tech', 'GOOGL': 'tech', 'NVDA': 'tech',
        'AMZN': 'tech', 'META': 'tech', 'TSLA': 'tech',
        'JPM': 'finance', 'V': 'finance', 'MA': 'finance',
        'JNJ': 'healthcare', 'UNH': 'healthcare',
        'SPY': 'etf', 'QQQ': 'etf',
        'TLT': 'bonds', 'GLD': 'commodities',
        # AI Infra — cybersecurity
        'CRWD': 'cybersecurity', 'PANW': 'cybersecurity',
        'ZS': 'cybersecurity', 'FTNT': 'cybersecurity',
        # AI Infra — data centers
        'EQIX': 'data_centers', 'DLR': 'data_centers',
        'AMT': 'data_centers', 'IREN': 'data_centers',
        # AI Infra — optics/semis
        'AMAT': 'optics', 'LRCX': 'optics', 'ONTO': 'optics',
        # AI Infra — robotics
        'ABB': 'robotics', 'ROK': 'robotics',
        # AI Infra — rare earth
        'MP': 'rare_earth', 'UUUU': 'rare_earth', 'LTHM': 'rare_earth',
    }

    def __init__(self):
        self._init_db()
        self._finbert = None  # Lazy-loaded
        self._finbert_lock = threading.Lock()

    def _init_db(self):
        conn = sqlite3.connect(SecureConfig.DB_PATH)
        c = conn.cursor()
        c.execute('''CREATE TABLE IF NOT EXISTS market_data (
            symbol TEXT, date TEXT, open REAL, high REAL, low REAL,
            close REAL, volume REAL, cached_at TEXT,
            PRIMARY KEY (symbol, date))''')
        c.execute('''CREATE TABLE IF NOT EXISTS macro_data (
            series TEXT, date TEXT, value REAL, cached_at TEXT,
            PRIMARY KEY (series, date))''')
        c.execute('''CREATE TABLE IF NOT EXISTS data_quality (
            symbol TEXT PRIMARY KEY, score REAL, last_check TEXT,
            missing_pct REAL, days_available INT)''')
        c.execute('''CREATE TABLE IF NOT EXISTS ai_ready_metadata (
            symbol TEXT PRIMARY KEY, threshold_met INTEGER,
            sentiment_score REAL, completeness REAL, processed_at TEXT)''')
        conn.commit()
        conn.close()

    # ---- Market data ----

    def fetch_market_data(self, symbol: str, period: str = '3y') -> Optional[pd.DataFrame]:
        cached = self._get_cached_market(symbol, period)
        if cached is not None:
            return cached
        for attempt in range(3):
            try:
                ticker = yf.Ticker(symbol)
                df = ticker.history(period=period, interval='1d', auto_adjust=True)
                if df.empty or len(df) < 100:
                    return None
                df.index = pd.to_datetime(df.index).tz_localize(None)
                df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
                df.dropna(subset=['Close'], inplace=True)
                self._cache_market_data(symbol, df)
                self._score_data_quality(symbol, df)
                return df
            except Exception as e:
                wait = 2 ** attempt
                logger.log(LogLevel.WARNING, f"Fetch {symbol} attempt {attempt+1} failed: {e}", "DATA")
                time.sleep(wait)
        return None

    def fetch_multiple(self, symbols: List[str], period: str = '3y') -> Dict[str, pd.DataFrame]:
        results = {}
        with ThreadPoolExecutor(max_workers=8) as ex:
            futures = {ex.submit(self.fetch_market_data, s, period): s for s in symbols}
            for fut in as_completed(futures):
                sym = futures[fut]
                try:
                    df = fut.result()
                    if df is not None:
                        results[sym] = df
                except Exception as e:
                    logger.log(LogLevel.WARNING, f"Failed {sym}: {e}", "DATA")
        return results

    def _get_cached_market(self, symbol: str, period: str) -> Optional[pd.DataFrame]:
        days = {'1y': 365, '2y': 730, '3y': 1095, '5y': 1825}.get(period, 1095)
        start = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            df = pd.read_sql_query(
                "SELECT * FROM market_data WHERE symbol=? AND date>=? ORDER BY date",
                conn, params=(symbol, start))
            conn.close()
            if df.empty:
                return None
            last_cached = pd.to_datetime(df['cached_at'].iloc[-1])
            if datetime.now() - last_cached > timedelta(seconds=SecureConfig.MARKET_CACHE_TTL):
                return None
            df['date'] = pd.to_datetime(df['date'])
            df.set_index('date', inplace=True)
            return df[['open', 'high', 'low', 'close', 'volume']].rename(
                columns={'open': 'Open', 'high': 'High', 'low': 'Low',
                         'close': 'Close', 'volume': 'Volume'})
        except Exception:
            return None

    def _cache_market_data(self, symbol: str, df: pd.DataFrame):
        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            now = datetime.now().isoformat()
            records = [(symbol, str(idx.date()), float(r['Open']), float(r['High']),
                        float(r['Low']), float(r['Close']), float(r['Volume']), now)
                       for idx, r in df.iterrows()]
            conn.executemany(
                "INSERT OR REPLACE INTO market_data VALUES (?,?,?,?,?,?,?,?)", records)
            conn.commit()
            conn.close()
        except Exception as e:
            logger.log(LogLevel.WARNING, f"Cache write failed for {symbol}: {e}", "DATA")

    def _score_data_quality(self, symbol: str, df: pd.DataFrame):
        try:
            missing_pct = df.isnull().mean().mean()
            score = 1.0 - missing_pct
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            conn.execute(
                "INSERT OR REPLACE INTO data_quality VALUES (?,?,?,?,?)",
                (symbol, float(score), datetime.now().isoformat(),
                 float(missing_pct), len(df)))
            conn.commit()
            conn.close()
        except Exception:
            pass

    def get_data_quality(self, symbol: str) -> float:
        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            row = conn.execute(
                "SELECT score FROM data_quality WHERE symbol=?", (symbol,)).fetchone()
            conn.close()
            return float(row[0]) if row else 0.5
        except Exception:
            return 0.5

    def get_sector(self, symbol: str) -> str:
        return self.SECTOR_MAP.get(symbol, 'unknown')

    # ---- Macro data (FRED) ----

    def fetch_macro_data(self) -> Dict[str, pd.Series]:
        if not SecureConfig.FRED_API_KEY or not REQUESTS_AVAILABLE:
            return self._simulate_macro()
        results = {}
        for series_id, name in self.FRED_SERIES.items():
            try:
                url = (f"https://api.stlouisfed.org/fred/series/observations"
                       f"?series_id={series_id}&api_key={SecureConfig.FRED_API_KEY}"
                       f"&file_type=json&sort_order=desc&limit=60")
                r = requests.get(url, timeout=10)
                if r.status_code == 200:
                    obs = r.json().get('observations', [])
                    data = {o['date']: float(o['value'])
                            for o in obs if o['value'] != '.'}
                    if data:
                        s = pd.Series(data)
                        s.index = pd.to_datetime(s.index)
                        results[name] = s.sort_index()
            except Exception as e:
                logger.log(LogLevel.WARNING, f"FRED {series_id}: {e}", "MACRO")
        if not results:
            return self._simulate_macro()
        return results

    def _simulate_macro(self) -> Dict[str, pd.Series]:
        dates = pd.date_range(end=datetime.now(), periods=60, freq='ME')
        np.random.seed(42)
        return {
            'CPI':          pd.Series(np.random.normal(3.5, 0.3, 60), index=dates),
            'GDP':          pd.Series(np.random.normal(2.5, 0.5, 60), index=dates),
            'Unemployment': pd.Series(np.random.normal(4.0, 0.2, 60), index=dates),
            'FedFunds':     pd.Series(np.random.normal(5.25, 0.1, 60), index=dates),
            'YieldSpread':  pd.Series(np.random.normal(-0.1, 0.3, 60), index=dates),
            'VIX_FRED':     pd.Series(np.random.normal(18.0, 4.0, 60), index=dates),
            'ConsumerSent': pd.Series(np.random.normal(70.0, 5.0, 60), index=dates),
            'IndustrialProd': pd.Series(np.random.normal(0.5, 0.2, 60), index=dates),
        }

    def fetch_geo_sentiment(self, symbol: str) -> Dict[str, Any]:
        """Simplified geo-political risk (extended with TextBlob if available)."""
        base_risks = {
            'AAPL': 0.08, 'NVDA': 0.12, 'TSLA': 0.15, 'META': 0.10,
            'JPM': 0.06, 'MSFT': 0.07, 'GOOGL': 0.09, 'AMZN': 0.08,
            'CRWD': 0.05, 'PANW': 0.05, 'EQIX': 0.04, 'DLR': 0.04,
            'AMAT': 0.14, 'LRCX': 0.14, 'MP': 0.20, 'UUUU': 0.18,
        }
        risk = base_risks.get(symbol, 0.10)
        return {'risk_score': risk, 'source': 'heuristic'}

    # ---- 4.4 NEW: AI-Ready Data Preparation ----

    def prepare_ai_ready_data(self, df: pd.DataFrame, symbol: str = '') -> pd.DataFrame:
        """
        AI-Ready Data Module (4.4):
        1. Completeness check vs threshold
        2. Outlier-robust cleaning
        3. FinBERT sentiment integration (if available)
        4. Metadata logging
        """
        logger.log(LogLevel.INFO, f"Preparing AI-ready data for {symbol}...", "AI-DATA")

        n_cols = len(df.columns)
        completeness = 1.0 - df.isnull().mean().mean()

        # Step 1: Drop rows that fail completeness threshold
        min_valid = int(SecureConfig.AI_READY_DATA_THRESHOLD * n_cols)
        df_clean = df.dropna(thresh=min_valid).copy()

        # Step 2: Fill remaining NaNs with forward-fill then interpolation
        df_clean = df_clean.ffill().bfill()
        df_clean = df_clean.interpolate(method='linear', limit_direction='both')

        # Step 3: Remove infinite values
        df_clean.replace([np.inf, -np.inf], np.nan, inplace=True)
        df_clean.fillna(0, inplace=True)

        # Step 4: Clip outliers (5-sigma clipping per column)
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            mean, std = df_clean[col].mean(), df_clean[col].std()
            if std > 0:
                df_clean[col] = df_clean[col].clip(mean - 5 * std, mean + 5 * std)

        # Step 5: FinBERT sentiment (if transformers available + news text in data)
        sentiment_score = 0.0
        if TRANSFORMERS_AVAILABLE and SecureConfig.DSLM_ENABLED:
            sentiment_score = self._get_finbert_sentiment(symbol)
            if sentiment_score != 0.0:
                df_clean['fin_sentiment'] = sentiment_score

        # Step 6: Log metadata to DB
        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            conn.execute(
                "INSERT OR REPLACE INTO ai_ready_metadata VALUES (?,?,?,?,?)",
                (symbol,
                 int(completeness >= SecureConfig.AI_READY_DATA_THRESHOLD),
                 float(sentiment_score),
                 float(completeness),
                 datetime.now().isoformat()))
            conn.commit()
            conn.close()
        except Exception:
            pass

        threshold_met = completeness >= SecureConfig.AI_READY_DATA_THRESHOLD
        logger.log(
            LogLevel.SUCCESS if threshold_met else LogLevel.WARNING,
            f"{symbol} AI-ready: completeness={completeness:.2%} "
            f"{'✓' if threshold_met else '✗ below threshold'} | "
            f"sentiment={sentiment_score:+.3f}",
            "AI-DATA"
        )
        return df_clean

    def _get_finbert_sentiment(self, symbol: str) -> float:
        """Lazy-load FinBERT and get sentiment for symbol-related text."""
        try:
            with self._finbert_lock:
                if self._finbert is None:
                    logger.log(LogLevel.INFO,
                               f"Loading FinBERT ({SecureConfig.FINBERT_MODEL})...", "AI-DATA")
                    self._finbert = hf_pipeline(
                        "sentiment-analysis",
                        model=SecureConfig.FINBERT_MODEL,
                        truncation=True,
                        max_length=512
                    )
            # Use a generic market-relevant sentence as proxy
            # In production this would use actual news headlines
            sample_text = f"{symbol} stock shows positive momentum and strong earnings growth"
            result = self._finbert(sample_text)[0]
            label = result['label'].upper()
            score = float(result['score'])
            if label == 'POSITIVE':
                return score
            elif label == 'NEGATIVE':
                return -score
            return 0.0
        except Exception as e:
            logger.log(LogLevel.WARNING, f"FinBERT error for {symbol}: {e}", "AI-DATA")
            return 0.0

    def build_macro_features(self, macro_data: Dict[str, pd.Series],
                              ref_date: pd.Timestamp) -> np.ndarray:
        features = []
        for name in ['CPI', 'GDP', 'Unemployment', 'FedFunds',
                     'YieldSpread', 'VIX_FRED', 'ConsumerSent', 'IndustrialProd']:
            series = macro_data.get(name)
            if series is not None and not series.empty:
                recent = series[series.index <= ref_date]
                if not recent.empty:
                    features.append(float(recent.iloc[-1]))
                    year_ago = series[series.index <= ref_date - timedelta(days=365)]
                    if not year_ago.empty:
                        yoy = (recent.iloc[-1] - year_ago.iloc[-1]) / (abs(year_ago.iloc[-1]) + 1e-9)
                        features.append(float(np.clip(yoy, -1, 1)))
                    else:
                        features.append(0.0)
                else:
                    features.extend([0.0, 0.0])
            else:
                features.extend([0.0, 0.0])
        return np.array(features, dtype=float)


# =============================================================================
# MODULE 4: ADAPTIVE FEATURE ENGINE (unchanged from 4.3)
# =============================================================================

class AdaptiveFeatureEngine:

    SECTOR_MAP = {
        'AAPL': 'tech', 'MSFT': 'tech', 'GOOGL': 'tech', 'NVDA': 'tech',
        'AMZN': 'tech', 'META': 'tech', 'TSLA': 'tech',
        'JPM': 'finance', 'V': 'finance', 'MA': 'finance',
        'JNJ': 'healthcare', 'UNH': 'healthcare',
        'SPY': 'etf', 'QQQ': 'etf',
        'TLT': 'bonds', 'GLD': 'commodities',
        'CRWD': 'cybersecurity', 'PANW': 'cybersecurity',
        'ZS': 'cybersecurity', 'FTNT': 'cybersecurity',
        'EQIX': 'data_centers', 'DLR': 'data_centers',
        'AMAT': 'optics', 'LRCX': 'optics',
        'ABB': 'robotics', 'ROK': 'robotics',
        'MP': 'rare_earth', 'UUUU': 'rare_earth',
    }

    def engineer_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        close  = df['Close']
        volume = df['Volume']

        df['Returns']    = close.pct_change()
        df['LogReturns'] = np.log(close / close.shift(1))
        df['Returns_5']  = close.pct_change(5)
        df['Returns_20'] = close.pct_change(20)

        for p in [20, 50, 200]:
            df[f'SMA_{p}'] = close.rolling(p).mean()
            df[f'EMA_{p}'] = close.ewm(span=p, adjust=False).mean()

        df['GoldenCross'] = (df['SMA_50'] > df['SMA_200']).astype(float)
        df['PriceSMA20']  = close / df['SMA_20'] - 1
        df['PriceSMA50']  = close / df['SMA_50'] - 1
        df['PriceSMA200'] = close / df['SMA_200'] - 1

        df['Vol_10']  = df['Returns'].rolling(10).std() * np.sqrt(252)
        df['Vol_20']  = df['Returns'].rolling(20).std() * np.sqrt(252)
        df['Vol_60']  = df['Returns'].rolling(60).std() * np.sqrt(252)
        df['VolRatio'] = df['Vol_10'] / (df['Vol_60'] + 1e-9)

        bb_mid = close.rolling(20).mean()
        bb_std = close.rolling(20).std()
        df['BB_Upper'] = bb_mid + 2 * bb_std
        df['BB_Lower'] = bb_mid - 2 * bb_std
        df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / (bb_mid + 1e-9)
        df['BB_Pos']   = (close - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'] + 1e-9)

        delta = close.diff()
        gain = delta.where(delta > 0, 0.0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
        rs = gain / (loss + 1e-9)
        df['RSI'] = 100 - 100 / (1 + rs)

        ema12 = close.ewm(span=12).mean()
        ema26 = close.ewm(span=26).mean()
        df['MACD']        = ema12 - ema26
        df['MACD_Signal'] = df['MACD'].ewm(span=9).mean()
        df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

        low14  = df['Low'].rolling(14).min()
        high14 = df['High'].rolling(14).max()
        df['Stoch_K'] = 100 * (close - low14) / (high14 - low14 + 1e-9)
        df['Stoch_D'] = df['Stoch_K'].rolling(3).mean()

        high_low  = df['High'] - df['Low']
        high_prev = (df['High'] - close.shift(1)).abs()
        low_prev  = (df['Low']  - close.shift(1)).abs()
        df['ATR'] = pd.concat([high_low, high_prev, low_prev], axis=1).max(axis=1).rolling(14).mean()

        df['Momentum_10'] = close / close.shift(10) - 1
        df['Momentum_20'] = close / close.shift(20) - 1
        df['Momentum_60'] = close / close.shift(60) - 1

        df['VolSMA20']    = volume.rolling(20).mean()
        df['VolumeRatio'] = volume / (df['VolSMA20'] + 1e-9)
        df['OBV']         = (np.sign(df['Returns']) * volume).cumsum()
        df['OBV_SMA20']   = df['OBV'].rolling(20).mean()
        df['OBV_Signal']  = (df['OBV'] > df['OBV_SMA20']).astype(float)

        df['DayOfWeek'] = df.index.dayofweek
        df['Month']     = df.index.month
        df['Quarter']   = df.index.quarter

        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.fillna(0, inplace=True)
        return df

    def get_feature_cols(self) -> List[str]:
        return [
            'Returns', 'LogReturns', 'Returns_5', 'Returns_20',
            'PriceSMA20', 'PriceSMA50', 'PriceSMA200', 'GoldenCross',
            'Vol_10', 'Vol_20', 'Vol_60', 'VolRatio',
            'BB_Width', 'BB_Pos',
            'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist',
            'Stoch_K', 'Stoch_D', 'ATR',
            'Momentum_10', 'Momentum_20', 'Momentum_60',
            'VolumeRatio', 'OBV_Signal',
        ]

    def get_sector(self, symbol: str) -> str:
        return self.SECTOR_MAP.get(symbol, 'unknown')


# =============================================================================
# MODULE 5: ADAPTIVE WEIGHT MANAGER (unchanged from 4.3)
# =============================================================================

class AdaptiveWeightManager:

    def __init__(self):
        self._performance: Dict[str, Dict[str, List[float]]] = {}
        self._weights: Dict[str, Dict[str, float]] = {}
        self._db_path = SecureConfig.DB_PATH
        self._init_db()
        self._load_from_db()

    def _init_db(self):
        conn = sqlite3.connect(self._db_path)
        conn.execute('''CREATE TABLE IF NOT EXISTS adaptive_weights (
            key TEXT PRIMARY KEY, weights_json TEXT, updated_at TEXT)''')
        conn.execute('''CREATE TABLE IF NOT EXISTS model_errors (
            key TEXT, model TEXT, error REAL, ts TEXT)''')
        conn.commit()
        conn.close()

    def _load_from_db(self):
        try:
            conn = sqlite3.connect(self._db_path)
            rows = conn.execute("SELECT key, weights_json FROM adaptive_weights").fetchall()
            conn.close()
            for key, wj in rows:
                self._weights[key] = json.loads(wj)
        except Exception:
            pass

    def _key(self, symbol: str, regime: str) -> str:
        return f"{symbol}_{regime}"

    def get_weights(self, symbol: str, regime: str,
                    available_models: List[str]) -> Dict[str, float]:
        key = self._key(symbol, regime)
        stored = self._weights.get(key, {})
        weights = {}
        for m in available_models:
            weights[m] = max(
                SecureConfig.MIN_MODEL_WEIGHT,
                stored.get(m, SecureConfig.INITIAL_ENSEMBLE_WEIGHTS.get(m, 0.10))
            )
        total = sum(weights.values())
        return {m: w / total for m, w in weights.items()}

    def record_error(self, symbol: str, regime: str, model: str, error: float):
        key = self._key(symbol, regime)
        if key not in self._performance:
            self._performance[key] = {}
        if model not in self._performance[key]:
            self._performance[key][model] = []
        self._performance[key][model].append(error)
        try:
            conn = sqlite3.connect(self._db_path)
            conn.execute("INSERT INTO model_errors VALUES (?,?,?,?)",
                         (key, model, error, datetime.now().isoformat()))
            conn.commit()
            conn.close()
        except Exception:
            pass
        total = sum(len(v) for v in self._performance[key].values())
        if total >= SecureConfig.WEIGHT_UPDATE_INTERVAL:
            self._update_weights(key)

    def _update_weights(self, key: str):
        perf = self._performance.get(key, {})
        if not perf:
            return
        scores = {}
        for model, errors in perf.items():
            if errors:
                mse = np.mean(np.array(errors) ** 2)
                scores[model] = 1.0 / (mse + 1e-8)
        if not scores:
            return
        T = SecureConfig.WEIGHT_TEMPERATURE
        vals = np.array(list(scores.values()))
        vals = vals / vals.max()
        exp_vals = np.exp(vals / T)
        softmax = exp_vals / exp_vals.sum()
        new_weights = {}
        for i, model in enumerate(scores.keys()):
            new_weights[model] = max(SecureConfig.MIN_MODEL_WEIGHT, float(softmax[i]))
        total = sum(new_weights.values())
        new_weights = {m: w / total for m, w in new_weights.items()}
        self._weights[key] = new_weights
        try:
            conn = sqlite3.connect(self._db_path)
            conn.execute("INSERT OR REPLACE INTO adaptive_weights VALUES (?,?,?)",
                         (key, json.dumps(new_weights), datetime.now().isoformat()))
            conn.commit()
            conn.close()
        except Exception:
            pass
        for model in self._performance.get(key, {}):
            self._performance[key][model] = self._performance[key][model][-50:]


# =============================================================================
# MODULE 6: ML ENSEMBLE (unchanged from 4.3 — TFT + Attention-LSTM + tree models)
# =============================================================================

class TemporalFusionTransformer:

    def __init__(self, lookback: int, n_features: int, horizon: int = 1):
        self.lookback = lookback
        self.n_features = n_features
        self.horizon = horizon
        self.model = None
        self.scaler = RobustScaler()
        self._trained = False

    def build(self) -> None:
        if not TF_AVAILABLE:
            return
        inputs = Input(shape=(self.lookback, self.n_features))
        grn = Dense(self.n_features, activation='elu')(inputs)
        gate = Dense(self.n_features, activation='sigmoid')(inputs)
        vsn_out = Multiply()([grn, gate])
        lstm_out = LSTM(128, return_sequences=True)(vsn_out)
        lstm_out = Dropout(0.2)(lstm_out)
        attn_out = MultiHeadAttention(num_heads=4, key_dim=32)(lstm_out, lstm_out)
        attn_out = Add()([lstm_out, attn_out])
        attn_out = LayerNormalization()(attn_out)
        decoder = LSTM(64, return_sequences=False)(attn_out)
        decoder = Dropout(0.2)(decoder)
        hidden = Dense(32, activation='relu')(decoder)
        out_mean = Dense(self.horizon, name='mean')(hidden)
        out_var  = Dense(self.horizon, activation='softplus', name='var')(hidden)
        self.model = Model(inputs=inputs, outputs=[out_mean, out_var])
        self.model.compile(
            optimizer=Adam(0.001),
            loss={'mean': 'mse', 'var': 'mse'},
            loss_weights={'mean': 1.0, 'var': 0.1}
        )

    def fit(self, X_train, y_train, X_val, y_val):
        if not TF_AVAILABLE:
            return
        if self.model is None:
            self.build()
        y_var = np.ones_like(y_train) * 0.01
        cbs = [
            EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
            ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-5)
        ]
        self.model.fit(
            X_train,
            {'mean': y_train, 'var': y_var},
            validation_data=(X_val, {'mean': y_val, 'var': np.ones_like(y_val) * 0.01}),
            epochs=SecureConfig.TFT_EPOCHS,
            batch_size=SecureConfig.BATCH_SIZE,
            callbacks=cbs, verbose=0
        )
        self._trained = True

    def predict(self, X) -> Tuple[float, float]:
        if not TF_AVAILABLE or not self._trained or self.model is None:
            return 0.0, 1.0
        try:
            out = self.model.predict(X[-1:], verbose=0)
            mean = float(out[0][0][0])
            var  = float(out[1][0][0])
            return mean, float(np.sqrt(max(var, 1e-8)))
        except Exception:
            return 0.0, 1.0


class AttentionLSTM:

    def __init__(self, lookback: int, n_features: int):
        self.lookback = lookback
        self.n_features = n_features
        self.model = None
        self._trained = False

    def build(self):
        if not TF_AVAILABLE:
            return
        inputs = Input(shape=(self.lookback, self.n_features))
        x = LSTM(128, return_sequences=True)(inputs)
        x = Dropout(0.2)(x)
        x = LSTM(64, return_sequences=True)(x)
        attn = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
        x = Add()([x, attn])
        x = LayerNormalization()(x)
        x = LSTM(32, return_sequences=False)(x)
        x = Dropout(0.2)(x)
        x = Dense(16, activation='relu')(x)
        out = Dense(1)(x)
        self.model = Model(inputs=inputs, outputs=out)
        self.model.compile(optimizer=Adam(0.001), loss='mse')

    def fit(self, X_train, y_train, X_val, y_val):
        if not TF_AVAILABLE:
            return
        if self.model is None:
            self.build()
        cbs = [EarlyStopping(patience=6, restore_best_weights=True)]
        self.model.fit(X_train, y_train,
                       validation_data=(X_val, y_val),
                       epochs=SecureConfig.LSTM_EPOCHS,
                       batch_size=SecureConfig.BATCH_SIZE,
                       callbacks=cbs, verbose=0)
        self._trained = True

    def predict(self, X) -> float:
        if not TF_AVAILABLE or not self._trained or self.model is None:
            return 0.0
        try:
            return float(self.model.predict(X[-1:], verbose=0)[0][0])
        except Exception:
            return 0.0


class AdvancedMLEnsemble:

    def __init__(self, weight_manager: AdaptiveWeightManager):
        self.weight_manager = weight_manager
        self.tft  = None
        self.lstm = None
        self.xgb_model  = None
        self.lgbm_model = None
        self.rf_model   = None
        self.scaler = RobustScaler()
        self._available: List[str] = []

    def prepare_sequences(self, df: pd.DataFrame,
                           feature_cols: List[str]) -> Tuple[np.ndarray, np.ndarray]:
        data = df[feature_cols].values.astype(float)
        data = self.scaler.fit_transform(data)
        close = df['Close'].values
        lookback = SecureConfig.LOOKBACK_WINDOW
        horizon  = SecureConfig.FORECAST_HORIZON
        X, y = [], []
        for i in range(lookback, len(data) - horizon):
            X.append(data[i - lookback: i])
            curr_price = close[i]
            fut_price  = close[i + horizon]
            if curr_price > 0:
                y.append((fut_price - curr_price) / curr_price)
            else:
                y.append(0.0)
        return np.array(X), np.array(y)

    def train(self, df: pd.DataFrame, feature_cols: List[str]) -> bool:
        logger.start_timer("ensemble_train")
        X, y = self.prepare_sequences(df, feature_cols)
        if len(X) < 80:
            return False
        n = len(X)
        s1, s2 = int(n * 0.70), int(n * 0.85)
        X_train, X_val = X[:s1], X[s1:s2]
        y_train, y_val = y[:s1], y[s1:s2]
        X_flat_train = X_train.reshape(X_train.shape[0], -1)
        self._available = []

        if TF_AVAILABLE:
            try:
                self.tft = TemporalFusionTransformer(X.shape[1], X.shape[2])
                self.tft.fit(X_train, y_train, X_val, y_val)
                self._available.append('tft')
            except Exception as e:
                logger.log(LogLevel.WARNING, f"TFT failed: {e}", "ML")

        if TF_AVAILABLE:
            try:
                self.lstm = AttentionLSTM(X.shape[1], X.shape[2])
                self.lstm.fit(X_train, y_train, X_val, y_val)
                self._available.append('lstm')
            except Exception as e:
                logger.log(LogLevel.WARNING, f"LSTM failed: {e}", "ML")

        if XGB_AVAILABLE:
            try:
                self.xgb_model = xgb.XGBRegressor(
                    n_estimators=200, max_depth=6, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8,
                    n_jobs=-1, verbosity=0, random_state=42)
                self.xgb_model.fit(X_flat_train, y_train,
                                   eval_set=[(X_val.reshape(X_val.shape[0], -1), y_val)],
                                   verbose=False)
                self._available.append('xgb')
            except Exception as e:
                logger.log(LogLevel.WARNING, f"XGBoost failed: {e}", "ML")

        if LGBM_AVAILABLE:
            try:
                self.lgbm_model = lgb.LGBMRegressor(
                    n_estimators=200, max_depth=6, learning_rate=0.05,
                    subsample=0.8, n_jobs=-1, verbosity=-1, random_state=42)
                self.lgbm_model.fit(X_flat_train, y_train,
                                    eval_set=[(X_val.reshape(X_val.shape[0], -1), y_val)])
                self._available.append('lgbm')
            except Exception as e:
                logger.log(LogLevel.WARNING, f"LightGBM failed: {e}", "ML")

        try:
            self.rf_model = RandomForestRegressor(
                n_estimators=200, max_depth=15,
                min_samples_leaf=3, n_jobs=-1, random_state=42)
            self.rf_model.fit(X_flat_train, y_train)
            self._available.append('rf')
        except Exception as e:
            logger.log(LogLevel.WARNING, f"RF failed: {e}", "ML")

        logger.end_timer("ensemble_train")
        logger.log(LogLevel.SUCCESS, f"Trained: {self._available}", "ML")
        return len(self._available) > 0

    def predict(self, df: pd.DataFrame, feature_cols: List[str],
                symbol: str, regime: str) -> Dict[str, Any]:
        if not self._available:
            return {'prediction': 0.0, 'confidence': 0.1, 'individual': {}, 'uncertainty': 1.0}
        X, _ = self.prepare_sequences(df, feature_cols)
        if len(X) == 0:
            return {'prediction': 0.0, 'confidence': 0.1, 'individual': {}, 'uncertainty': 1.0}
        individual = {}
        X_flat = X[-1:].reshape(1, -1)
        if 'tft' in self._available and self.tft:
            mean, _ = self.tft.predict(X)
            individual['tft'] = mean
        if 'lstm' in self._available and self.lstm:
            individual['lstm'] = self.lstm.predict(X)
        if 'xgb' in self._available and self.xgb_model:
            individual['xgb'] = float(self.xgb_model.predict(X_flat)[0])
        if 'lgbm' in self._available and self.lgbm_model:
            individual['lgbm'] = float(self.lgbm_model.predict(X_flat)[0])
        if 'rf' in self._available and self.rf_model:
            individual['rf'] = float(self.rf_model.predict(X_flat)[0])
        if not individual:
            return {'prediction': 0.0, 'confidence': 0.1, 'individual': {}, 'uncertainty': 1.0}
        weights = self.weight_manager.get_weights(symbol, regime, list(individual.keys()))
        pred = sum(individual[m] * weights.get(m, 0.0) for m in individual)
        preds_arr = np.array(list(individual.values()))
        uncertainty = float(np.std(preds_arr))
        confidence  = float(np.clip(1.0 / (1.0 + uncertainty * 10), 0.05, 0.95))
        return {
            'prediction':  float(pred),
            'confidence':  confidence,
            'uncertainty': uncertainty,
            'individual':  individual,
            'weights_used': weights,
        }

    def get_feature_importance(self) -> Dict[str, np.ndarray]:
        imp = {}
        if self.rf_model is not None:
            imp['rf'] = self.rf_model.feature_importances_
        if XGB_AVAILABLE and self.xgb_model is not None:
            imp['xgb'] = self.xgb_model.feature_importances_
        return imp


# =============================================================================
# MODULE 7: REGIME DETECTOR (unchanged from 4.3)
# =============================================================================

class RegimeAwarePredictor:

    REGIMES = ['bull', 'bear', 'sideways', 'crisis']

    def __init__(self):
        self.gmm = GaussianMixture(n_components=3, random_state=42, max_iter=200)
        self._gmm_fitted = False
        self._regime_models: Dict[str, Any] = {}
        self._scaler = StandardScaler()

    def detect(self, spy_df: Optional[pd.DataFrame]) -> Dict[str, Any]:
        if spy_df is None or len(spy_df) < 200:
            return self._unknown_regime()
        returns = spy_df['Close'].pct_change().dropna()
        vol_20  = returns.rolling(20).std() * np.sqrt(252)
        mom_20  = spy_df['Close'] / spy_df['Close'].shift(20) - 1
        sma50   = spy_df['Close'].rolling(50).mean()
        sma200  = spy_df['Close'].rolling(200).mean()

        features_df = pd.DataFrame({
            'returns': returns,
            'vol_20': vol_20,
            'mom_20': mom_20,
        }).dropna()

        if len(features_df) < 100:
            return self._unknown_regime()

        try:
            X = self._scaler.fit_transform(features_df)
            self.gmm.fit(X)
            self._gmm_fitted = True
            labels = self.gmm.predict(X)
            probs  = self.gmm.predict_proba(X[-1:]).flatten()

            cluster_stats = {}
            for i in range(3):
                mask = labels == i
                if mask.sum() > 0:
                    cluster_stats[i] = {
                        'mean_return': features_df['returns'][mask].mean(),
                        'mean_vol': features_df['vol_20'][mask].mean(),
                    }

            # Map GMM clusters to regimes
            sorted_by_return = sorted(cluster_stats.items(),
                                      key=lambda x: x[1]['mean_return'])
            regime_map = {}
            if len(sorted_by_return) >= 3:
                regime_map[sorted_by_return[0][0]] = 'bear'
                regime_map[sorted_by_return[1][0]] = 'sideways'
                regime_map[sorted_by_return[2][0]] = 'bull'
            elif len(sorted_by_return) == 2:
                regime_map[sorted_by_return[0][0]] = 'bear'
                regime_map[sorted_by_return[1][0]] = 'bull'

            # Current cluster
            current_cluster = self.gmm.predict(X[-1:])[0]
            base_regime = regime_map.get(current_cluster, 'sideways')

            # Crisis override via recent volatility
            recent_vol = features_df['vol_20'].iloc[-1]
            recent_mom = features_df['mom_20'].iloc[-1]
            crisis_mode = (recent_vol > 0.35 or recent_mom < -0.12)
            if crisis_mode:
                base_regime = 'crisis'

            # Build probability dict across 4 regimes
            regime_probs = {'bull': 0.0, 'bear': 0.0, 'sideways': 0.0, 'crisis': 0.0}
            for cluster_id, prob in enumerate(probs):
                r = regime_map.get(cluster_id, 'sideways')
                regime_probs[r] = max(regime_probs[r], float(prob))
            if crisis_mode:
                regime_probs['crisis'] = max(regime_probs['crisis'], 0.70)
                total = sum(regime_probs.values())
                if total > 0:
                    regime_probs = {k: v / total for k, v in regime_probs.items()}

            return {
                'regime':      base_regime,
                'probabilities': regime_probs,
                'crisis_mode': crisis_mode,
                'recent_vol':  float(recent_vol),
                'recent_mom':  float(recent_mom),
            }
        except Exception as e:
            logger.log(LogLevel.WARNING, f"Regime detection failed: {e}", "REGIME")
            return self._unknown_regime()

    def _unknown_regime(self) -> Dict[str, Any]:
        return {
            'regime': 'sideways',
            'probabilities': {'bull': 0.25, 'bear': 0.25, 'sideways': 0.40, 'crisis': 0.10},
            'crisis_mode': False,
            'recent_vol': 0.15,
            'recent_mom': 0.0,
        }

    def train_regime_models(self, spy_df: pd.DataFrame, regime_info: Dict,
                             feature_cols: List[str]) -> None:
        """Train regime-specific models for blended prediction."""
        regime = regime_info.get('regime', 'sideways')
        if regime not in self._regime_models:
            try:
                returns = spy_df['Close'].pct_change().dropna()
                X_flat  = spy_df[feature_cols].dropna().values
                y_flat  = returns.reindex(spy_df.dropna().index).fillna(0).values
                min_len = min(len(X_flat), len(y_flat))
                if min_len > 50:
                    model = GradientBoostingRegressor(
                        n_estimators=100, max_depth=4, random_state=42)
                    model.fit(X_flat[:min_len], y_flat[:min_len])
                    self._regime_models[regime] = model
            except Exception:
                pass

    def blended_prediction(self, df: pd.DataFrame,
                            regime_probs: Dict[str, float],
                            feature_cols: List[str]) -> Optional[float]:
        if not self._regime_models:
            return None
        try:
            X = df[feature_cols].iloc[-1:].values
            pred = 0.0
            total_w = 0.0
            for regime, model in self._regime_models.items():
                w = regime_probs.get(regime, 0.0)
                pred += float(model.predict(X)[0]) * w
                total_w += w
            return pred / total_w if total_w > 0 else None
        except Exception:
            return None


# =============================================================================
# MODULE 8: CALIBRATOR (unchanged from 4.3)
# =============================================================================

class AdaptiveCalibrator:

    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._global_model: Optional[IsotonicRegression] = None
        self._symbol_models: Dict[str, IsotonicRegression] = {}
        self._regime_models: Dict[str, IsotonicRegression] = {}
        self._init_db()

    def _init_db(self):
        conn = sqlite3.connect(self.db_path)
        conn.execute('''CREATE TABLE IF NOT EXISTS calibration_data (
            symbol TEXT, regime TEXT, raw_conf REAL, outcome INT, ts TEXT)''')
        conn.commit()
        conn.close()

    def calibrate_confidence(self, raw_conf: float, symbol: str, regime: str) -> float:
        cal_conf = raw_conf
        if self._symbol_models.get(symbol):
            try:
                cal_conf = float(self._symbol_models[symbol].predict([[cal_conf]])[0])
            except Exception:
                pass
        elif self._regime_models.get(regime):
            try:
                cal_conf = float(self._regime_models[regime].predict([[cal_conf]])[0])
            except Exception:
                pass
        elif self._global_model:
            try:
                cal_conf = float(self._global_model.predict([[cal_conf]])[0])
            except Exception:
                pass
        return float(np.clip(cal_conf, 0.01, 0.99))

    def record_outcome(self, symbol: str, regime: str,
                       raw_conf: float, outcome: int):
        try:
            conn = sqlite3.connect(self.db_path)
            conn.execute("INSERT INTO calibration_data VALUES (?,?,?,?,?)",
                         (symbol, regime, raw_conf, outcome,
                          datetime.now().isoformat()))
            conn.commit()
            conn.close()
        except Exception:
            pass

    def should_update(self) -> bool:
        try:
            conn = sqlite3.connect(self.db_path)
            count = conn.execute("SELECT COUNT(*) FROM calibration_data").fetchone()[0]
            conn.close()
            return count >= SecureConfig.MIN_CALIBRATION_SAMPLES
        except Exception:
            return False

    def calibrate(self):
        try:
            conn = sqlite3.connect(self.db_path)
            df = pd.read_sql_query("SELECT * FROM calibration_data", conn)
            conn.close()
            if len(df) < SecureConfig.MIN_CALIBRATION_SAMPLES:
                return
            X = df['raw_conf'].values.reshape(-1, 1)
            y = df['outcome'].values.astype(float)
            self._global_model = IsotonicRegression(out_of_bounds='clip')
            self._global_model.fit(X.flatten(), y)
        except Exception as e:
            logger.log(LogLevel.WARNING, f"Calibration failed: {e}", "CALIBRATION")


# =============================================================================
# MODULE 9: DECISION ENGINE (unchanged from 4.3)
# =============================================================================

@dataclass
class Decision:
    symbol: str
    decision: str
    raw_confidence: float
    calibrated_confidence: float
    raw_prediction: float
    model_disagreement: float
    market_stress: float
    regime: str
    sector: str
    data_quality: float
    reason: str
    would_have_traded: bool
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    counterfactual: Optional[str] = None
    agent_flag: bool = False          # 4.4: flagged by agentic prototype
    anomaly_score: float = 0.0        # 4.4: anomaly score from agent


class MarketStressIndicator:

    def calculate(self, df: pd.DataFrame,
                  regime_info: Dict[str, Any]) -> float:
        try:
            returns  = df['Close'].pct_change().dropna().tail(20)
            vol_20   = returns.std() * np.sqrt(252)
            vol_norm = float(np.clip(vol_20 / 0.40, 0, 1))
            crisis   = 0.8 if regime_info.get('crisis_mode') else 0.0
            rsi      = float(df['RSI'].iloc[-1]) if 'RSI' in df.columns else 50.0
            rsi_stress = float(abs(rsi - 50) / 50)
            stress = 0.50 * vol_norm + 0.30 * crisis + 0.20 * rsi_stress
            return float(np.clip(stress, 0, 1))
        except Exception:
            return 0.3


class PerformanceTracker:

    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        conn = sqlite3.connect(self.db_path)
        conn.execute('''CREATE TABLE IF NOT EXISTS predictions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            symbol TEXT, predicted_return REAL, confidence REAL,
            calibrated_confidence REAL, model_disagreement REAL,
            market_stress REAL, decision TEXT, would_have_traded INTEGER,
            regime TEXT, ts TEXT)''')
        conn.execute('''CREATE TABLE IF NOT EXISTS decisions_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            symbol TEXT, decision TEXT, confidence REAL, reason TEXT,
            agent_flag INTEGER, anomaly_score REAL, ts TEXT)''')
        conn.commit()
        conn.close()

    def record_prediction(self, symbol: str, predicted_return: float,
                           confidence: float, calibrated_confidence: float,
                           model_disagreement: float, market_stress: float,
                           decision: str, would_have_traded: bool, regime: str):
        try:
            conn = sqlite3.connect(self.db_path)
            conn.execute(
                "INSERT INTO predictions VALUES (NULL,?,?,?,?,?,?,?,?,?,?)",
                (symbol, predicted_return, confidence, calibrated_confidence,
                 model_disagreement, market_stress, decision,
                 int(would_have_traded), regime, datetime.now().isoformat()))
            conn.commit()
            conn.close()
        except Exception:
            pass

    def record_decision(self, d: Decision):
        try:
            conn = sqlite3.connect(self.db_path)
            conn.execute(
                "INSERT INTO decisions_log VALUES (NULL,?,?,?,?,?,?,?)",
                (d.symbol, d.decision, d.calibrated_confidence, d.reason,
                 int(d.agent_flag), float(d.anomaly_score),
                 datetime.now().isoformat()))
            conn.commit()
            conn.close()
        except Exception:
            pass

    def get_summary(self) -> Dict[str, Any]:
        try:
            conn = sqlite3.connect(self.db_path)
            df = pd.read_sql_query("SELECT * FROM predictions", conn)
            conn.close()
            if df.empty:
                return self._empty_summary()
            traded = df[df['would_have_traded'] == 1]
            correct  = int((traded['predicted_return'] * traded['calibrated_confidence'] > 0).sum())
            wrong    = int(len(traded) - correct)
            missed   = int((df['would_have_traded'] == 0).sum())
            neutral  = int((df['decision'] == 'HOLD').sum())
            tv = (correct * SecureConfig.OUTCOME_VALUES['correct_trade'] +
                  wrong   * SecureConfig.OUTCOME_VALUES['wrong_trade'] +
                  missed  * SecureConfig.OUTCOME_VALUES['missed_opportunity'])
            return {'correct': correct, 'wrong': wrong,
                    'missed': missed, 'neutral': neutral, 'total_value': tv}
        except Exception:
            return self._empty_summary()

    def _empty_summary(self):
        return {'correct': 0, 'wrong': 0, 'missed': 0, 'neutral': 0, 'total_value': 0.0}


class IntelligentDecisionEngine:

    def __init__(self, calibrator: AdaptiveCalibrator,
                 stress_ind: MarketStressIndicator,
                 tracker: PerformanceTracker):
        self.calibrator = calibrator
        self.stress_ind = stress_ind
        self.tracker    = tracker

    def decide(self, symbol: str, raw_prediction: float, raw_confidence: float,
               individual_preds: Dict[str, float], weights_used: Dict[str, float],
               market_data: pd.DataFrame, regime_info: Dict[str, Any],
               sector: str, data_quality: float,
               agent_flag: bool = False, anomaly_score: float = 0.0) -> Decision:

        regime = regime_info.get('regime', 'sideways')
        cal_conf = self.calibrator.calibrate_confidence(raw_confidence, symbol, regime)
        stress   = self.stress_ind.calculate(market_data, regime_info)

        model_disagreement = (float(np.std(list(individual_preds.values())))
                              if len(individual_preds) > 1 else 0.0)

        # Volume check
        vol_ratio = (float(market_data['VolumeRatio'].iloc[-1])
                     if 'VolumeRatio' in market_data.columns else 1.0)

        # Decision gates
        reasons = []
        would_have_traded = True

        if cal_conf < SecureConfig.MIN_CONFIDENCE:
            reasons.append(f"conf {cal_conf:.2f} < {SecureConfig.MIN_CONFIDENCE}")
            would_have_traded = False
        if model_disagreement > SecureConfig.MAX_MODEL_DISAGREEMENT:
            reasons.append(f"model disagreement {model_disagreement:.4f}")
            would_have_traded = False
        if stress > SecureConfig.MAX_STRESS_INDEX:
            reasons.append(f"market stress {stress:.2f}")
            would_have_traded = False
        if data_quality < 0.50:
            reasons.append(f"low data quality {data_quality:.2f}")
            would_have_traded = False
        if vol_ratio < SecureConfig.MIN_VOLUME_RATIO:
            reasons.append(f"low volume ratio {vol_ratio:.2f}")
            would_have_traded = False

        # Agent override: if agent flagged anomaly, force HOLD
        if agent_flag and anomaly_score > SecureConfig.AGENT_ANOMALY_THRESHOLD:
            reasons.append(f"🤖 agent anomaly detected (z={anomaly_score:.2f})")
            would_have_traded = False

        # Primary decision
        if not would_have_traded:
            decision_str = 'HOLD'
            reason = "HOLD — gates: " + "; ".join(reasons)
        elif raw_prediction > 0.02:
            decision_str = 'BUY'
            reason = f"BUY — pred={raw_prediction*100:.1f}%, conf={cal_conf:.2f}"
        elif raw_prediction < -0.02:
            decision_str = 'SELL'
            reason = f"SELL — pred={raw_prediction*100:.1f}%, conf={cal_conf:.2f}"
        else:
            decision_str = 'HOLD'
            reason = f"HOLD — prediction near-zero ({raw_prediction*100:.2f}%)"

        # Crisis override
        if regime_info.get('crisis_mode') and decision_str == 'BUY':
            decision_str = 'HOLD'
            reason += " [CRISIS OVERRIDE → HOLD]"
            would_have_traded = False

        # Counterfactual
        counterfactual = None
        if decision_str == 'HOLD' and not agent_flag:
            if raw_prediction > 0.03:
                counterfactual = (f"Would BUY if confidence ≥ {SecureConfig.MIN_CONFIDENCE:.2f} "
                                  f"and stress ≤ {SecureConfig.MAX_STRESS_INDEX:.2f}")
            elif raw_prediction < -0.03:
                counterfactual = (f"Would SELL if confidence ≥ {SecureConfig.MIN_CONFIDENCE:.2f}")

        return Decision(
            symbol=symbol,
            decision=decision_str,
            raw_confidence=raw_confidence,
            calibrated_confidence=cal_conf,
            raw_prediction=raw_prediction,
            model_disagreement=model_disagreement,
            market_stress=stress,
            regime=regime,
            sector=sector,
            data_quality=data_quality,
            reason=reason,
            would_have_traded=would_have_traded,
            counterfactual=counterfactual,
            agent_flag=agent_flag,
            anomaly_score=anomaly_score,
        )


# =============================================================================
# MODULE 10: PORTFOLIO OPTIMIZER (unchanged from 4.3)
# =============================================================================

class PortfolioOptimizerV2:

    def __init__(self, capital: float, risk_aversion: float = 2.5):
        self.capital = capital
        self.risk_aversion = risk_aversion

    def optimize(self, decisions: Dict[str, Decision],
                 market_data: Dict[str, pd.DataFrame],
                 regime_info: Dict[str, Any]) -> Dict[str, float]:
        buy_assets = [s for s, d in decisions.items() if d.decision == 'BUY']
        if not buy_assets:
            logger.log(LogLevel.WARNING, "No BUY decisions — equal weight fallback", "OPTIMIZER")
            hold_assets = [s for s in decisions.keys() if s in market_data]
            return self._equal_weight(hold_assets[:5]) if hold_assets else {}

        assets_with_data = [a for a in buy_assets if a in market_data]
        if not assets_with_data:
            return {}

        is_crisis = regime_info.get('crisis_mode', False)

        try:
            returns_df = pd.DataFrame({
                a: market_data[a]['Close'].pct_change().dropna()
                for a in assets_with_data
            }).dropna()

            if len(returns_df) < 60:
                return self._equal_weight(assets_with_data)

            n = len(assets_with_data)
            mu  = returns_df.mean().values * 252
            cov = returns_df.cov().values * 252

            # Black-Litterman views
            views = np.array([decisions[a].raw_prediction for a in assets_with_data])
            view_conf = np.array([decisions[a].calibrated_confidence for a in assets_with_data])
            bl_mu = mu + view_conf * (views * 252 - mu)

            def bl_objective(w):
                ret = np.dot(w, bl_mu)
                vol = np.sqrt(np.dot(w.T, np.dot(cov, w)) + 1e-9)
                return -(ret - self.risk_aversion * vol ** 2)

            def rp_objective(w):
                port_vol = np.sqrt(np.dot(w.T, np.dot(cov, w)))
                marginal = np.dot(cov, w) / (port_vol + 1e-9)
                risk_contributions = w * marginal
                target_rc = port_vol / n
                return float(np.sum((risk_contributions - target_rc) ** 2))

            constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]
            max_pos = (SecureConfig.CRISIS_MAX_POSITION if is_crisis
                       else SecureConfig.MAX_POSITION_SIZE)
            bounds = tuple((SecureConfig.MIN_POSITION_SIZE, max_pos) for _ in range(n))
            x0 = np.array([1.0 / n] * n)

            regime = regime_info.get('regime', 'sideways')
            if is_crisis:
                bl_w, rp_w = 0.0, 1.0
            elif regime == 'bull':
                bl_w, rp_w = 0.70, 0.30
            elif regime == 'bear':
                bl_w, rp_w = 0.30, 0.70
            else:
                bl_w, rp_w = 0.50, 0.50

            try:
                bl_result  = minimize(bl_objective, x0, method='SLSQP',
                                      bounds=bounds, constraints=constraints,
                                      options={'maxiter': 500})
                bl_weights = bl_result.x
            except Exception:
                bl_weights = x0

            try:
                rp_result  = minimize(rp_objective, x0, method='SLSQP',
                                      bounds=bounds, constraints=constraints,
                                      options={'maxiter': 500})
                rp_weights = rp_result.x
            except Exception:
                rp_weights = x0

            final_weights = bl_w * bl_weights + rp_w * rp_weights
            final_weights = np.clip(final_weights, SecureConfig.MIN_POSITION_SIZE, max_pos)
            final_weights /= final_weights.sum()

            allocation = {}
            for i, asset in enumerate(assets_with_data):
                if final_weights[i] >= SecureConfig.MIN_POSITION_SIZE:
                    allocation[asset] = float(final_weights[i]) * self.capital
            return allocation

        except Exception as e:
            logger.log(LogLevel.WARNING, f"Optimizer failed: {e} — using equal weight", "OPTIMIZER")
            return self._equal_weight(assets_with_data)

    def _equal_weight(self, assets: List[str]) -> Dict[str, float]:
        if not assets:
            return {}
        w = 1.0 / len(assets)
        return {a: w * self.capital for a in assets}


# =============================================================================
# MODULE 11 (NEW 4.4): AGENTIC PROTOTYPE
# =============================================================================

class AgenticTask:
    """Represents a discrete task executed by an AI agent."""

    def __init__(self, task_id: str, task_type: str,
                 data: Dict[str, Any], callback: Optional[Callable] = None):
        self.task_id   = task_id
        self.task_type = task_type
        self.data      = data
        self.callback  = callback
        self.result: Optional[Dict] = None
        self.error: Optional[str]   = None
        self.executed_at: Optional[str] = None


class AgenticPrototype:
    """
    4.4 Core New Module: Autonomous AI Agent Prototype.

    Agents:
    1. AnomalyDetectionAgent  — detects statistical anomalies in market data
    2. PortfolioRebalanceAgent — auto-flags positions that drift beyond threshold
    3. DataQualityAgent       — monitors and validates AI-ready data pipelines
    4. TopDownStrategyAgent   — enterprise-wide portfolio strategy overview

    Uses concurrent.futures for parallel execution.
    Integrates with IntelligentDecisionEngine via agent_flag + anomaly_score.
    """

    TASK_TYPES = [
        'anomaly_detection',
        'portfolio_rebalance',
        'data_quality_check',
        'top_down_strategy',
        'init_portfolio',
    ]

    def __init__(self, config: type = SecureConfig):
        self.config = config
        self._task_history: List[AgenticTask] = []
        self._anomaly_detector = IsolationForest(
            contamination=0.05, random_state=42, n_jobs=-1)
        self._anomaly_fitted = False
        self._envelope: Optional[EllipticEnvelope] = None
        self._lock = threading.Lock()
        self._init_db()
        logger.log(LogLevel.AGENT, "AgenticPrototype initialized", "AGENT")

    def _init_db(self):
        conn = sqlite3.connect(SecureConfig.DB_PATH)
        conn.execute('''CREATE TABLE IF NOT EXISTS agent_tasks (
            task_id TEXT PRIMARY KEY,
            task_type TEXT,
            result_json TEXT,
            error TEXT,
            executed_at TEXT)''')
        conn.commit()
        conn.close()

    def execute_task(self, task_type: str,
                     data: Optional[Dict] = None) -> Dict[str, Any]:
        """
        Execute a named agent task with timeout protection.
        Returns result dict with 'status', 'output', and optional 'flags'.
        """
        if not self.config.AGENTIC_ENABLED:
            return {'status': 'disabled', 'output': {}}
        if task_type not in self.TASK_TYPES:
            return {'status': 'error', 'output': {}, 'error': f"Unknown task: {task_type}"}

        data = data or {}
        task_id = f"{task_type}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
        task = AgenticTask(task_id=task_id, task_type=task_type, data=data)

        logger.log(LogLevel.AGENT, f"Executing agent task: {task_type} [{task_id}]", "AGENT")
        logger.start_timer(f"agent_{task_id}")

        handler_map = {
            'anomaly_detection':  self._agent_anomaly_detection,
            'portfolio_rebalance': self._agent_portfolio_rebalance,
            'data_quality_check': self._agent_data_quality,
            'top_down_strategy':  self._agent_top_down_strategy,
            'init_portfolio':     self._agent_init_portfolio,
        }

        try:
            with ThreadPoolExecutor(max_workers=1) as ex:
                future = ex.submit(handler_map[task_type], data)
                result = future.result(timeout=self.config.AGENT_TASK_TIMEOUT)
            task.result = result
            task.executed_at = datetime.now().isoformat()
            status = 'success'
        except Exception as e:
            task.error = str(e)
            result = {'status': 'error', 'output': {}, 'error': str(e)}
            status = 'error'
            logger.log(LogLevel.WARNING, f"Agent task {task_type} failed: {e}", "AGENT")

        with self._lock:
            self._task_history.append(task)

        self._persist_task(task_id, task_type,
                           json.dumps(result), task.error or '', task.executed_at or '')
        logger.end_timer(f"agent_{task_id}")
        return result

    # ---- Agent handlers ----

    def _agent_anomaly_detection(self, data: Dict) -> Dict:
        """
        AnomalyDetectionAgent:
        Uses IsolationForest + EllipticEnvelope to detect outlier returns.
        Returns per-symbol anomaly scores and flags.
        """
        market_data: Dict[str, pd.DataFrame] = data.get('market_data', {})
        if not market_data:
            return {'status': 'no_data', 'output': {}, 'flags': {}}

        anomaly_results = {}
        flagged_symbols = {}

        for symbol, df in market_data.items():
            try:
                if len(df) < 60:
                    continue

                returns = df['Close'].pct_change().dropna().values.reshape(-1, 1)
                vol_20  = pd.Series(returns.flatten()).rolling(20).std().dropna().values

                # Fit IsolationForest on returns + vol
                X = np.column_stack([
                    returns[-len(vol_20):].flatten(),
                    vol_20
                ])

                if len(X) < 30:
                    continue

                iso = IsolationForest(contamination=0.05, random_state=42)
                iso.fit(X[:-1])  # Fit on historical, score on latest

                # Anomaly score: more negative = more anomalous
                raw_score = float(iso.decision_function(X[-1:].reshape(1, -1))[0])
                # Z-score of the latest return
                z_score = float(abs(returns[-1][0]) / (np.std(returns) + 1e-9))

                # Normalize to 0-5 scale (higher = more anomalous)
                anomaly_score = float(np.clip((1 - raw_score) * 2.5 + z_score * 0.5, 0, 5))

                anomaly_results[symbol] = {
                    'anomaly_score':   anomaly_score,
                    'z_score':         z_score,
                    'isolation_score': raw_score,
                    'is_anomaly':      anomaly_score > self.config.AGENT_ANOMALY_THRESHOLD
                }

                if anomaly_score > self.config.AGENT_ANOMALY_THRESHOLD:
                    flagged_symbols[symbol] = anomaly_score
                    logger.log(LogLevel.AGENT,
                               f"⚡ ANOMALY: {symbol} — score={anomaly_score:.2f}, "
                               f"z={z_score:.2f}", "AGENT")

            except Exception as e:
                logger.log(LogLevel.WARNING, f"Anomaly check failed for {symbol}: {e}", "AGENT")

        return {
            'status':          'success',
            'output':          anomaly_results,
            'flags':           flagged_symbols,
            'flagged_count':   len(flagged_symbols),
            'analyzed_count':  len(anomaly_results),
        }

    def _agent_portfolio_rebalance(self, data: Dict) -> Dict:
        """
        PortfolioRebalanceAgent:
        Checks if current allocation has drifted > AGENT_REBALANCE_TRIGGER
        from target allocation and flags assets requiring rebalancing.
        """
        current_alloc: Dict[str, float] = data.get('current_allocation', {})
        target_alloc:  Dict[str, float] = data.get('target_allocation', {})

        if not current_alloc or not target_alloc:
            return {'status': 'no_data', 'output': {}, 'rebalance_needed': False}

        total_current = sum(current_alloc.values()) or 1.0
        total_target  = sum(target_alloc.values()) or 1.0

        rebalance_flags = {}
        adjustments = {}

        for symbol in set(list(current_alloc.keys()) + list(target_alloc.keys())):
            curr_pct = current_alloc.get(symbol, 0.0) / total_current
            tgt_pct  = target_alloc.get(symbol, 0.0) / total_target
            drift    = curr_pct - tgt_pct

            if abs(drift) > self.config.AGENT_REBALANCE_TRIGGER:
                rebalance_flags[symbol] = {
                    'drift':          drift,
                    'current_pct':    curr_pct,
                    'target_pct':     tgt_pct,
                    'action':         'SELL' if drift > 0 else 'BUY',
                    'magnitude':      min(abs(drift), self.config.AGENT_MAX_AUTO_ADJUST)
                }
                adjustments[symbol] = tgt_pct - curr_pct

        rebalance_needed = len(rebalance_flags) > 0
        if rebalance_needed:
            logger.log(LogLevel.AGENT,
                       f"Portfolio rebalance needed: {len(rebalance_flags)} assets flagged",
                       "AGENT")

        return {
            'status':            'success',
            'output':            rebalance_flags,
            'adjustments':       adjustments,
            'rebalance_needed':  rebalance_needed,
            'flagged_count':     len(rebalance_flags),
        }

    def _agent_data_quality(self, data: Dict) -> Dict:
        """
        DataQualityAgent:
        Validates AI-ready data quality across loaded market data.
        Flags assets with quality below threshold.
        """
        market_data: Dict[str, pd.DataFrame] = data.get('market_data', {})
        threshold = self.config.AI_READY_DATA_THRESHOLD

        quality_report = {}
        failed_assets  = []

        for symbol, df in market_data.items():
            completeness = 1.0 - df.isnull().mean().mean()
            has_enough_history = len(df) >= SecureConfig.MIN_HISTORY_DAYS

            # Stationarity check (ADF-lite via variance ratio)
            try:
                returns = df['Close'].pct_change().dropna()
                variance_ratio = (returns.head(len(returns)//2).var() /
                                  (returns.tail(len(returns)//2).var() + 1e-9))
                variance_stable = 0.3 < variance_ratio < 3.0
            except Exception:
                variance_stable = True

            quality_score = (0.5 * completeness +
                             0.3 * float(has_enough_history) +
                             0.2 * float(variance_stable))

            quality_report[symbol] = {
                'completeness':      completeness,
                'has_history':       has_enough_history,
                'variance_stable':   variance_stable,
                'quality_score':     quality_score,
                'threshold_met':     quality_score >= threshold,
                'rows':              len(df),
            }

            if quality_score < threshold:
                failed_assets.append(symbol)

        logger.log(LogLevel.AGENT,
                   f"Data quality: {len(market_data) - len(failed_assets)}/{len(market_data)} "
                   f"assets meet threshold ({threshold:.0%})", "AGENT")

        return {
            'status':       'success',
            'output':       quality_report,
            'failed':       failed_assets,
            'pass_rate':    (len(market_data) - len(failed_assets)) / max(len(market_data), 1),
        }

    def _agent_top_down_strategy(self, data: Dict) -> Dict:
        """
        TopDownStrategyAgent (Enterprise Mode):
        Synthesizes macro + regime + allocation into a top-down AI strategy brief.
        Acts as Chief AI Officer strategy layer.
        """
        regime_info: Dict = data.get('regime_info', {})
        macro_data:  Dict = data.get('macro_data', {})
        decisions:   Dict = data.get('decisions', {})
        allocation:  Dict = data.get('allocation', {})

        regime = regime_info.get('regime', 'unknown')
        crisis = regime_info.get('crisis_mode', False)

        buy_count  = sum(1 for d in decisions.values() if hasattr(d, 'decision') and d.decision == 'BUY')
        sell_count = sum(1 for d in decisions.values() if hasattr(d, 'decision') and d.decision == 'SELL')
        hold_count = sum(1 for d in decisions.values() if hasattr(d, 'decision') and d.decision == 'HOLD')

        # Macro signals
        vix    = float(macro_data.get('VIX_FRED', pd.Series([18.0])).iloc[-1]) if macro_data.get('VIX_FRED') is not None else 18.0
        fed    = float(macro_data.get('FedFunds', pd.Series([5.0])).iloc[-1]) if macro_data.get('FedFunds') is not None else 5.0
        spread = float(macro_data.get('YieldSpread', pd.Series([0.0])).iloc[-1]) if macro_data.get('YieldSpread') is not None else 0.0

        # Strategic guidance
        if crisis:
            strategic_stance = "DEFENSIVE"
            strategic_rationale = (f"Crisis regime detected. VIX={vix:.1f}, "
                                   f"yield_spread={spread:.2f}. "
                                   "Maximize cash, reduce equity exposure, prioritize bonds.")
        elif regime == 'bull':
            strategic_stance = "GROWTH"
            strategic_rationale = (f"Bull regime. Fed={fed:.2f}%, "
                                   f"VIX={vix:.1f}. "
                                   "Overweight AI infra + tech. Consider momentum strategies.")
        elif regime == 'bear':
            strategic_stance = "CAUTIOUS"
            strategic_rationale = (f"Bear regime. Yield spread={spread:.2f}. "
                                   "Defensive positioning. Quality over growth. "
                                   "Cybersecurity and bonds as safe harbor.")
        else:
            strategic_stance = "BALANCED"
            strategic_rationale = (f"Sideways regime. "
                                   "Diversification across AI infra + core equities.")

        top_buys = sorted(
            [(s, d) for s, d in decisions.items()
             if hasattr(d, 'decision') and d.decision == 'BUY'],
            key=lambda x: x[1].calibrated_confidence, reverse=True
        )[:5]

        return {
            'status': 'success',
            'output': {
                'strategic_stance':    strategic_stance,
                'strategic_rationale': strategic_rationale,
                'regime':              regime,
                'crisis_mode':         crisis,
                'vix':                 vix,
                'fed_funds_rate':      fed,
                'yield_spread':        spread,
                'buy_count':           buy_count,
                'sell_count':          sell_count,
                'hold_count':          hold_count,
                'top_buy_signals':     [(s, d.calibrated_confidence) for s, d in top_buys],
                'total_allocated':     sum(allocation.values()),
            }
        }

    def _agent_init_portfolio(self, data: Dict) -> Dict:
        """
        InitPortfolioAgent:
        Runs on startup — validates universe, checks for existing positions,
        prepares initial agent state.
        """
        symbols = data.get('symbols', [])
        capital = data.get('capital', 0.0)

        logger.log(LogLevel.AGENT,
                   f"Initializing portfolio agent: {len(symbols)} symbols, "
                   f"capital={capital:,.0f}", "AGENT")

        return {
            'status': 'success',
            'output': {
                'initialized':     True,
                'symbols_checked': len(symbols),
                'capital':         capital,
                'timestamp':       datetime.now().isoformat(),
                'agentic_version': '4.4.0',
            }
        }

    def get_anomaly_flags(self, market_data: Dict[str, pd.DataFrame]) -> Dict[str, float]:
        """Convenience: run anomaly detection and return symbol → score dict."""
        result = self.execute_task('anomaly_detection', {'market_data': market_data})
        return result.get('flags', {})

    def _persist_task(self, task_id: str, task_type: str,
                      result_json: str, error: str, executed_at: str):
        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            conn.execute(
                "INSERT OR REPLACE INTO agent_tasks VALUES (?,?,?,?,?)",
                (task_id, task_type, result_json[:4000], error, executed_at))
            conn.commit()
            conn.close()
        except Exception:
            pass

    def get_task_history(self) -> List[Dict]:
        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            rows = conn.execute(
                "SELECT task_id, task_type, result_json, error, executed_at "
                "FROM agent_tasks ORDER BY executed_at DESC LIMIT 50"
            ).fetchall()
            conn.close()
            return [{'task_id': r[0], 'task_type': r[1],
                     'executed_at': r[4], 'error': r[3]} for r in rows]
        except Exception:
            return []


# =============================================================================
# MODULE 12 (NEW 4.4): DIVERSIFICATION MODULE — AI INFRASTRUCTURE
# =============================================================================

class DiversificationModule:
    """
    4.4 New Module: AI Infrastructure Diversification.

    Manages a curated universe of AI-related instruments across:
    - Cybersecurity (protection of AI models)
    - Data Centers (compute infrastructure)
    - Optics/Semis (fiber, photonics for data transfer)
    - Robotics (physical AI)
    - Rare Earth (raw materials for AI hardware)

    Uses LP optimization (PuLP if available, scipy fallback) to allocate
    within AI infra while respecting MAX_AI_INFRA_EXPOSURE.
    """

    def __init__(self, capital: float):
        self.capital = capital
        self._category_weights: Dict[str, float] = {}

    def get_ai_infra_universe(self) -> Dict[str, List[str]]:
        return SecureConfig.AI_INFRA_INSTRUMENTS

    def score_category(self, category: str,
                       market_data: Dict[str, pd.DataFrame]) -> float:
        """
        Score an AI infra category based on recent momentum of its instruments.
        Returns average 20-day momentum across available instruments.
        """
        instruments = SecureConfig.AI_INFRA_INSTRUMENTS.get(category, [])
        scores = []
        for sym in instruments:
            df = market_data.get(sym)
            if df is not None and len(df) > 25:
                mom_20 = float(df['Close'].iloc[-1] / df['Close'].iloc[-21] - 1)
                scores.append(mom_20)
        return float(np.mean(scores)) if scores else 0.0

    def optimize_ai_infra_allocation(
            self,
            market_data: Dict[str, pd.DataFrame],
            decisions: Dict[str, Decision],
            base_allocation: Dict[str, float]) -> Dict[str, float]:
        """
        Compute AI infra allocation as complement to base portfolio.
        Respects MAX_AI_INFRA_EXPOSURE and per-category limits.
        """
        max_ai_budget = self.capital * SecureConfig.MAX_AI_INFRA_EXPOSURE
        current_base  = sum(base_allocation.values())
        remaining     = max(0.0, self.capital - current_base)
        ai_budget     = min(max_ai_budget, remaining)

        if ai_budget < 100:  # Not enough room
            logger.log(LogLevel.INFO,
                       "No budget for AI infra allocation — base fills portfolio", "DIVERSIFY")
            return base_allocation

        categories = SecureConfig.AI_INFRA_CATEGORIES
        category_scores = {cat: self.score_category(cat, market_data)
                           for cat in categories}

        # Score-weighted allocation across categories
        positive_cats = {c: max(s, 0.001) for c, s in category_scores.items()}
        total_score = sum(positive_cats.values()) or 1.0
        category_weights = {c: s / total_score for c, s in positive_cats.items()}

        ai_allocation = {}

        # If PuLP available — LP optimization for best within-category instrument
        if PULP_AVAILABLE:
            ai_allocation = self._lp_optimize(
                categories, category_weights, ai_budget, market_data, decisions)
        else:
            ai_allocation = self._heuristic_optimize(
                categories, category_weights, ai_budget, market_data, decisions)

        # Merge with base allocation
        merged = dict(base_allocation)
        for sym, amt in ai_allocation.items():
            if amt >= 50:  # Min meaningful position
                merged[sym] = merged.get(sym, 0.0) + amt

        self._category_weights = category_weights
        logger.log(LogLevel.SUCCESS,
                   f"AI infra allocation: {len(ai_allocation)} instruments, "
                   f"total={sum(ai_allocation.values()):,.0f}", "DIVERSIFY")
        return merged

    def _lp_optimize(self, categories: List[str],
                     category_weights: Dict[str, float],
                     ai_budget: float,
                     market_data: Dict[str, pd.DataFrame],
                     decisions: Dict[str, Decision]) -> Dict[str, float]:
        """PuLP-based LP allocation within AI infra budget."""
        allocation = {}
        per_cat_budget = {cat: ai_budget * w for cat, w in category_weights.items()}

        for cat in categories:
            cat_budget = per_cat_budget.get(cat, 0.0)
            if cat_budget < 50:
                continue
            instruments = [sym for sym in SecureConfig.AI_INFRA_INSTRUMENTS.get(cat, [])
                           if sym in market_data]
            if not instruments:
                continue

            try:
                prob = pulp.LpProblem(f"ai_infra_{cat}", pulp.LpMaximize)
                n = len(instruments)
                w_vars = [pulp.LpVariable(f"w_{sym}", lowBound=0, upBound=1)
                          for sym in instruments]

                # Objective: maximize momentum-weighted return
                momentums = []
                for sym in instruments:
                    df = market_data.get(sym)
                    if df is not None and len(df) > 25:
                        mom = float(df['Close'].iloc[-1] / df['Close'].iloc[-21] - 1)
                    else:
                        mom = 0.0
                    momentums.append(mom)

                prob += pulp.lpSum(w_vars[i] * momentums[i] for i in range(n))
                prob += pulp.lpSum(w_vars) == 1  # weights sum to 1

                # Concentration constraint
                for wv in w_vars:
                    prob += wv <= 0.6

                prob.solve(pulp.PULP_CBC_CMD(msg=0))

                if pulp.LpStatus[prob.status] == 'Optimal':
                    for i, sym in enumerate(instruments):
                        alloc = float(w_vars[i].value() or 0) * cat_budget
                        if alloc >= 50:
                            allocation[sym] = alloc
                else:
                    # Equal weight fallback
                    per_sym = cat_budget / len(instruments)
                    for sym in instruments:
                        allocation[sym] = per_sym
            except Exception as e:
                logger.log(LogLevel.WARNING, f"LP failed for {cat}: {e} — equal weight", "DIVERSIFY")
                if instruments:
                    per_sym = cat_budget / len(instruments)
                    for sym in instruments:
                        allocation[sym] = per_sym

        return allocation

    def _heuristic_optimize(self, categories: List[str],
                             category_weights: Dict[str, float],
                             ai_budget: float,
                             market_data: Dict[str, pd.DataFrame],
                             decisions: Dict[str, Decision]) -> Dict[str, float]:
        """Momentum-weighted heuristic allocation (no PuLP needed)."""
        allocation = {}
        per_cat_budget = {cat: ai_budget * w for cat, w in category_weights.items()}

        for cat in categories:
            cat_budget = per_cat_budget.get(cat, 0.0)
            if cat_budget < 50:
                continue
            instruments = [sym for sym in SecureConfig.AI_INFRA_INSTRUMENTS.get(cat, [])
                           if sym in market_data]
            if not instruments:
                continue

            # Score by momentum, pick top 2
            scored = []
            for sym in instruments:
                df = market_data.get(sym)
                if df is not None and len(df) > 25:
                    mom = float(df['Close'].iloc[-1] / df['Close'].iloc[-21] - 1)
                    scored.append((sym, mom))
            scored.sort(key=lambda x: -x[1])
            top = scored[:2] if len(scored) >= 2 else scored

            if top:
                total_mom = sum(max(m, 0.001) for _, m in top)
                for sym, mom in top:
                    w = max(mom, 0.001) / total_mom
                    allocation[sym] = w * cat_budget

        return allocation

    def print_ai_infra_report(self, allocation: Dict[str, float],
                               market_data: Dict[str, pd.DataFrame]):
        """Print AI infrastructure allocation breakdown."""
        W = 100
        print(f"\n{'─' * W}")
        print(f"  🏗️  AI INFRASTRUCTURE DIVERSIFICATION REPORT")
        print(f"{'─' * W}")

        # Group by category
        all_ai_syms = {}
        for cat, syms in SecureConfig.AI_INFRA_INSTRUMENTS.items():
            for sym in syms:
                all_ai_syms[sym] = cat

        by_cat: Dict[str, Dict[str, float]] = {}
        for sym, amt in allocation.items():
            cat = all_ai_syms.get(sym)
            if cat:
                if cat not in by_cat:
                    by_cat[cat] = {}
                by_cat[cat][sym] = amt

        if not by_cat:
            print("  No AI infrastructure instruments in current allocation.")
            print(f"{'─' * W}\n")
            return

        cat_icons = {
            'cybersecurity': '🛡️ ',
            'data_centers':  '🏢',
            'optics':        '💡',
            'robotics':      '🤖',
            'rare_earth':    '⛏️ ',
        }

        total_ai = sum(amt for syms in by_cat.values() for amt in syms.values())
        print(f"  Total AI Infra Allocation: €{total_ai:,.0f} "
              f"({total_ai / max(self.capital, 1) * 100:.1f}% of portfolio)")
        print()

        for cat, syms in by_cat.items():
            icon = cat_icons.get(cat, '📦')
            cat_total = sum(syms.values())
            print(f"  {icon}  {cat.upper().replace('_', ' ')} — €{cat_total:,.0f}")
            for sym, amt in sorted(syms.items(), key=lambda x: -x[1]):
                mom_20 = 0.0
                df = market_data.get(sym)
                if df is not None and len(df) > 25:
                    mom_20 = float(df['Close'].iloc[-1] / df['Close'].iloc[-21] - 1)
                pct = amt / self.capital * 100
                print(f"      {sym:<8} €{amt:>10,.0f}  ({pct:.1f}%)  "
                      f"20d mom: {mom_20:+.1%}")

        print(f"{'─' * W}\n")


# =============================================================================
# MODULE 13 (NEW 4.4): INTEGRITY LAYER
# =============================================================================

class IntegrityLayer:
    """
    4.4 New Module: AI Security + Decision Audit.

    1. Model drift detection (prediction distribution shift)
    2. Graph-based anomaly detection in prediction correlations (if networkx)
    3. Decision trace logging for full audit
    4. Model poisoning heuristics
    """

    def __init__(self):
        self._prediction_history: Dict[str, List[float]] = {}
        self._init_db()
        logger.log(LogLevel.INTEGRITY, "IntegrityLayer initialized", "INTEGRITY")

    def _init_db(self):
        conn = sqlite3.connect(SecureConfig.DB_PATH)
        conn.execute('''CREATE TABLE IF NOT EXISTS decision_traces (
            trace_id TEXT PRIMARY KEY,
            symbol TEXT,
            decision TEXT,
            prediction REAL,
            confidence REAL,
            anomaly_score REAL,
            agent_flag INTEGER,
            integrity_ok INTEGER,
            drift_score REAL,
            poison_suspected INTEGER,
            timestamp TEXT)''')
        conn.execute('''CREATE TABLE IF NOT EXISTS integrity_alerts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            alert_type TEXT,
            symbol TEXT,
            details TEXT,
            severity TEXT,
            timestamp TEXT)''')
        conn.commit()
        conn.close()

    def validate_model(self, symbol: str,
                       predictions: List[float],
                       model_name: str = 'ensemble') -> Dict[str, Any]:
        """
        Heuristic model validation:
        1. Distribution drift vs historical
        2. Outlier ratio (poisoning proxy)
        3. Correlation collapse check (via networkx if available)
        """
        result = {
            'symbol':            symbol,
            'model':             model_name,
            'drift_score':       0.0,
            'outlier_ratio':     0.0,
            'poison_suspected':  False,
            'integrity_ok':      True,
            'details':           [],
        }

        if not predictions or len(predictions) < 5:
            return result

        hist = self._prediction_history.get(symbol, [])
        arr  = np.array(predictions)

        # 1. Drift detection via KS test
        if len(hist) >= 20:
            hist_arr = np.array(hist[-100:])
            try:
                ks_stat, p_val = stats.ks_2samp(hist_arr, arr)
                drift_score = float(ks_stat)
                result['drift_score'] = drift_score
                if drift_score > SecureConfig.MODEL_DRIFT_THRESHOLD:
                    result['details'].append(
                        f"⚠️ Distribution drift detected (KS={ks_stat:.3f}, p={p_val:.4f})")
                    result['integrity_ok'] = False
                    self._log_alert('model_drift', symbol,
                                    f"KS={ks_stat:.3f}", 'MEDIUM')
            except Exception:
                pass

        # 2. Outlier ratio (poisoning heuristic)
        mean, std = arr.mean(), arr.std()
        outliers = np.abs(arr - mean) > 4 * std
        outlier_ratio = float(outliers.mean())
        result['outlier_ratio'] = outlier_ratio
        if outlier_ratio > 0.15:
            result['details'].append(
                f"🔴 High outlier ratio ({outlier_ratio:.1%}) — possible poisoning")
            result['poison_suspected'] = True
            result['integrity_ok'] = False
            self._log_alert('possible_poisoning', symbol,
                            f"outlier_ratio={outlier_ratio:.2f}", 'HIGH')

        # 3. Graph-based correlation anomaly (networkx)
        if NETWORKX_AVAILABLE and len(predictions) > 10:
            try:
                G = nx.Graph()
                for i in range(len(predictions)):
                    G.add_node(i, pred=predictions[i])
                for i in range(len(predictions) - 1):
                    corr = abs(predictions[i] - predictions[i+1])
                    if corr < 0.01:   # near-identical predictions = suspicious
                        G.add_edge(i, i+1, weight=corr)

                # High clustering = suspiciously correlated predictions
                if G.number_of_edges() > 0:
                    clustering = nx.average_clustering(G)
                    if clustering > 0.80:
                        result['details'].append(
                            f"🔴 High prediction clustering (networkx: {clustering:.2f}) "
                            "— possible data leak or poisoning")
                        result['poison_suspected'] = True
                        result['integrity_ok'] = False
            except Exception:
                pass

        # Update history
        if symbol not in self._prediction_history:
            self._prediction_history[symbol] = []
        self._prediction_history[symbol].extend(predictions)
        self._prediction_history[symbol] = self._prediction_history[symbol][-500:]

        if SecureConfig.POISONING_DETECTION_ENABLED and result['poison_suspected']:
            logger.log(LogLevel.INTEGRITY,
                       f"🔴 INTEGRITY ALERT: {symbol} — {result['details']}", "INTEGRITY")

        return result

    def log_decision_trace(self, decision: Decision,
                            integrity_result: Optional[Dict] = None) -> str:
        """Save full decision trace to DB for audit."""
        trace_id = hashlib.md5(
            f"{decision.symbol}{decision.timestamp}{decision.decision}".encode()
        ).hexdigest()

        integrity_ok   = int(integrity_result.get('integrity_ok', True)) if integrity_result else 1
        drift_score    = float(integrity_result.get('drift_score', 0.0)) if integrity_result else 0.0
        poison_susp    = int(integrity_result.get('poison_suspected', False)) if integrity_result else 0

        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            conn.execute(
                "INSERT OR REPLACE INTO decision_traces VALUES (?,?,?,?,?,?,?,?,?,?,?)",
                (trace_id, decision.symbol, decision.decision,
                 decision.raw_prediction, decision.calibrated_confidence,
                 decision.anomaly_score, int(decision.agent_flag),
                 integrity_ok, drift_score, poison_susp,
                 decision.timestamp))
            conn.commit()
            conn.close()
        except Exception as e:
            logger.log(LogLevel.WARNING, f"Trace log failed: {e}", "INTEGRITY")

        # Also write to TRACES_DIR as JSON
        try:
            trace_file = SecureConfig.TRACES_DIR / f"{trace_id}.json"
            trace_data = {
                'trace_id':     trace_id,
                'symbol':       decision.symbol,
                'decision':     decision.decision,
                'prediction':   decision.raw_prediction,
                'confidence':   decision.calibrated_confidence,
                'anomaly_score': decision.anomaly_score,
                'agent_flag':   decision.agent_flag,
                'reason':       decision.reason,
                'integrity':    integrity_result or {},
                'timestamp':    decision.timestamp,
            }
            trace_file.write_text(json.dumps(trace_data, indent=2))
        except Exception:
            pass

        return trace_id

    def _log_alert(self, alert_type: str, symbol: str, details: str, severity: str):
        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            conn.execute(
                "INSERT INTO integrity_alerts VALUES (NULL,?,?,?,?,?)",
                (alert_type, symbol, details, severity, datetime.now().isoformat()))
            conn.commit()
            conn.close()
        except Exception:
            pass

    def get_integrity_summary(self) -> Dict[str, Any]:
        try:
            conn = sqlite3.connect(SecureConfig.DB_PATH)
            total   = conn.execute("SELECT COUNT(*) FROM decision_traces").fetchone()[0]
            failed  = conn.execute(
                "SELECT COUNT(*) FROM decision_traces WHERE integrity_ok=0").fetchone()[0]
            poison  = conn.execute(
                "SELECT COUNT(*) FROM decision_traces WHERE poison_suspected=1").fetchone()[0]
            alerts  = conn.execute(
                "SELECT COUNT(*) FROM integrity_alerts").fetchone()[0]
            conn.close()
            return {
                'total_traced': total,
                'failed':       failed,
                'poisoning_suspected': poison,
                'alerts':       alerts,
                'integrity_rate': (total - failed) / max(total, 1),
            }
        except Exception:
            return {'total_traced': 0, 'failed': 0, 'integrity_rate': 1.0}


# =============================================================================
# MODULE 14 (NEW 4.4): ROI TRACKER + BUBBLE ANALYSIS
# =============================================================================

class ROITracker:
    """
    4.4 New Module: AI Investment ROI Tracker.

    Tracks:
    - Predicted vs actual returns per symbol
    - AI infra category performance vs SPY benchmark
    - Bubble warning when AI sector P/E multiples > threshold
    - Productivity-over-pilots metric (have AI investments generated alpha?)
    """

    def __init__(self):
        self._init_db()
        self._roi_history: Dict[str, List[Dict]] = {}

    def _init_db(self):
        conn = sqlite3.connect(SecureConfig.DB_PATH)
        conn.execute('''CREATE TABLE IF NOT EXISTS roi_tracking (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            symbol TEXT, category TEXT,
            predicted_return REAL, actual_return REAL,
            roi_delta REAL, mse REAL,
            bubble_warning INTEGER,
            timestamp TEXT)''')
        conn.commit()
        conn.close()

    def calculate_ai_roi(self, predictions: Dict[str, float],
                          actual_returns: Dict[str, float],
                          symbol_categories: Dict[str, str]) -> Dict[str, Any]:
        """
        Calculate ROI delta: predicted vs actual.
        Flags bubble if AI sector returns deviate significantly from predictions.
        """
        results = {}
        all_preds, all_actuals = [], []
        bubble_symbols = []

        for symbol in set(predictions.keys()) & set(actual_returns.keys()):
            pred   = predictions[symbol]
            actual = actual_returns[symbol]
            delta  = actual - pred
            mse    = float(mean_squared_error([actual], [pred]))
            cat    = symbol_categories.get(symbol, 'unknown')

            # Bubble heuristic: large positive prediction but flat/negative actual
            bubble_warning = (pred > 0.05 and actual < -0.02)

            results[symbol] = {
                'predicted':       pred,
                'actual':          actual,
                'delta':           delta,
                'mse':             mse,
                'category':        cat,
                'bubble_warning':  bubble_warning,
            }

            if bubble_warning:
                bubble_symbols.append(symbol)
                logger.log(LogLevel.ROI,
                           f"🫧 BUBBLE WARNING: {symbol} predicted {pred:+.1%} "
                           f"but actual {actual:+.1%}", "ROI")

            all_preds.append(pred)
            all_actuals.append(actual)

            # Persist
            try:
                conn = sqlite3.connect(SecureConfig.DB_PATH)
                conn.execute(
                    "INSERT INTO roi_tracking VALUES (NULL,?,?,?,?,?,?,?,?)",
                    (symbol, cat, pred, actual, delta, mse,
                     int(bubble_warning), datetime.now().isoformat()))
                conn.commit()
                conn.close()
            except Exception:
                pass

        # Overall metrics
        overall_mse = float(mean_squared_error(all_actuals, all_preds)) if all_actuals else 0.0
        alpha = float(np.mean(all_actuals) - np.mean(all_preds)) if all_actuals else 0.0
        productivity_score = float(np.clip(1.0 - overall_mse * 100, 0, 1))

        summary = {
            'per_symbol':          results,
            'overall_mse':         overall_mse,
            'alpha':               alpha,
            'productivity_score':  productivity_score,
            'bubble_count':        len(bubble_symbols),
            'bubble_symbols':      bubble_symbols,
            'n_tracked':           len(results),
            'ai_generating_alpha': alpha > 0,
        }

        logger.log(LogLevel.ROI,
                   f"ROI: MSE={overall_mse:.6f}, alpha={alpha:+.2%}, "
                   f"productivity={productivity_score:.2%}, "
                   f"bubble_warnings={len(bubble_symbols)}", "ROI")
        return summary

    def compare_to_benchmark(self, market_data: Dict[str, pd.DataFrame],
                              allocation: Dict[str, float],
                              lookback_days: int = 30) -> Dict[str, float]:
        """Compare portfolio return vs SPY benchmark over lookback period."""
        spy_df = market_data.get(SecureConfig.ROI_BENCHMARK_SYMBOL)
        if spy_df is None or len(spy_df) < lookback_days:
            return {'portfolio_return': 0.0, 'benchmark_return': 0.0, 'alpha': 0.0}

        # SPY return
        spy_return = float(
            spy_df['Close'].iloc[-1] / spy_df['Close'].iloc[-lookback_days] - 1
        )

        # Portfolio weighted return
        total_alloc = sum(allocation.values()) or 1.0
        portfolio_return = 0.0
        for sym, amt in allocation.items():
            df = market_data.get(sym)
            if df is not None and len(df) >= lookback_days:
                ret = float(df['Close'].iloc[-1] / df['Close'].iloc[-lookback_days] - 1)
                portfolio_return += (amt / total_alloc) * ret

        alpha = portfolio_return - spy_return
        return {
            'portfolio_return': portfolio_return,
            'benchmark_return': spy_return,
            'alpha':            alpha,
            'outperforming':    alpha > 0,
        }

    def print_roi_report(self, roi_summary: Dict[str, Any],
                          benchmark: Dict[str, float]):
        W = 100
        print(f"\n{'─' * W}")
        print(f"  💰 ROI TRACKER & BUBBLE ANALYSIS")
        print(f"{'─' * W}")
        print(f"  Productivity Score:     {roi_summary['productivity_score']:.1%}")
        print(f"  Overall MSE:            {roi_summary['overall_mse']:.6f}")
        print(f"  Alpha vs Predictions:   {roi_summary['alpha']:+.2%}")
        print(f"  Symbols Tracked:        {roi_summary['n_tracked']}")
        print(f"  AI Generating Alpha:    {'✅ YES' if roi_summary['ai_generating_alpha'] else '❌ NO'}")
        print()
        print(f"  Benchmark Comparison ({SecureConfig.ROI_BENCHMARK_SYMBOL}, 30d):")
        print(f"    Portfolio Return:   {benchmark['portfolio_return']:+.2%}")
        print(f"    SPY Return:         {benchmark['benchmark_return']:+.2%}")
        print(f"    Alpha:              {benchmark['alpha']:+.2%}  "
              f"{'✅ Outperforming' if benchmark.get('outperforming') else '❌ Underperforming'}")
        if roi_summary['bubble_count'] > 0:
            print()
            print(f"  ⚠️  BUBBLE WARNINGS ({roi_summary['bubble_count']}):")
            for sym in roi_summary['bubble_symbols']:
                d = roi_summary['per_symbol'].get(sym, {})
                print(f"    {sym}: pred={d.get('predicted', 0):+.1%}, "
                      f"actual={d.get('actual', 0):+.1%}")
        else:
            print(f"\n  ✅ No bubble warnings detected.")
        print(f"{'─' * W}\n")


# =============================================================================
# MODULE 15 (NEW 4.4): CHIEF AI OFFICER MODE
# =============================================================================

class ChiefAIOfficerMode:
    """
    Optional enterprise mode: simulates a Chief AI Officer dashboard.
    Synthesizes all 4.4 modules into a strategic enterprise briefing.
    Inspired by 2026 trend of AI leadership roles requiring cross-functional AI oversight.
    """

    def __init__(self, enabled: bool = False):
        self.enabled = enabled

    def generate_briefing(self,
                           agent_strategy: Dict,
                           integrity_summary: Dict,
                           roi_summary: Dict,
                           regime_info: Dict,
                           ai_infra_allocation: Dict,
                           capital: float) -> str:
        if not self.enabled:
            return ""

        lines = []
        W = 100
        lines.append("\n" + "█" * W)
        lines.append("  🏛️  CHIEF AI OFFICER BRIEFING — SmartInvest 4.4")
        lines.append("  Enterprise AI Strategy Dashboard")
        lines.append("█" * W)

        # Strategic stance
        strategy_out = agent_strategy.get('output', {})
        stance    = strategy_out.get('strategic_stance', 'UNKNOWN')
        rationale = strategy_out.get('strategic_rationale', '')
        lines.append(f"\n  📊 STRATEGIC STANCE: {stance}")
        lines.append(f"  {rationale}")

        # Market context
        lines.append(f"\n  🌐 MARKET CONTEXT:")
        lines.append(f"    Regime:          {regime_info.get('regime', '?').upper()}")
        lines.append(f"    Crisis Mode:     {'⚠️ YES' if regime_info.get('crisis_mode') else '✅ NO'}")
        lines.append(f"    VIX:             {strategy_out.get('vix', 0):.1f}")
        lines.append(f"    Fed Funds:       {strategy_out.get('fed_funds_rate', 0):.2f}%")
        lines.append(f"    Yield Spread:    {strategy_out.get('yield_spread', 0):.3f}")

        # AI Infra exposure
        ai_total = sum(ai_infra_allocation.values()) if ai_infra_allocation else 0
        lines.append(f"\n  🏗️  AI INFRASTRUCTURE EXPOSURE:")
        lines.append(f"    AI Infra Capital: €{ai_total:,.0f} ({ai_total/max(capital,1)*100:.1f}%)")

        # Decision summary
        lines.append(f"\n  📈 DECISION SUMMARY:")
        lines.append(f"    BUY:  {strategy_out.get('buy_count', 0)}")
        lines.append(f"    SELL: {strategy_out.get('sell_count', 0)}")
        lines.append(f"    HOLD: {strategy_out.get('hold_count', 0)}")

        top_buys = strategy_out.get('top_buy_signals', [])
        if top_buys:
            lines.append(f"\n  🎯 TOP BUY SIGNALS:")
            for sym, conf in top_buys[:3]:
                lines.append(f"    {sym:<8} confidence: {conf:.2%}")

        # Integrity
        lines.append(f"\n  🛡️  INTEGRITY STATUS:")
        lines.append(f"    Decisions Traced:  {integrity_summary.get('total_traced', 0)}")
        lines.append(f"    Integrity Rate:    {integrity_summary.get('integrity_rate', 1.0):.1%}")
        if integrity_summary.get('poisoning_suspected', 0) > 0:
            lines.append(f"    ⚠️  Poisoning Suspected: {integrity_summary['poisoning_suspected']} case(s)")

        # ROI
        lines.append(f"\n  💰 AI ROI METRICS:")
        lines.append(f"    Productivity Score: {roi_summary.get('productivity_score', 0):.1%}")
        lines.append(f"    Alpha (vs preds):   {roi_summary.get('alpha', 0):+.2%}")
        if roi_summary.get('bubble_count', 0) > 0:
            lines.append(f"    ⚠️  Bubble Warnings: {roi_summary['bubble_count']}")

        lines.append("\n" + "─" * W)
        return "\n".join(lines)


# =============================================================================
# MAIN ORCHESTRATOR: SmartInvest44
# =============================================================================

class SmartInvest44:
    """
    Main orchestrator for SmartInvest 4.4 — Agentic Prototype Edition.
    Builds on all 4.3 modules + 4.4 extensions:
      AgenticPrototype, DiversificationModule, IntegrityLayer, ROITracker,
      ChiefAIOfficerMode, AIReadyDataModule (in RobustDataManager).
    """

    VERSION  = "4.4.0"
    CODENAME = "Agentic Prototype"

    # Core universe (4.3) + AI infra universe (4.4)
    CORE_SYMBOLS = [
        'AAPL', 'MSFT', 'GOOGL', 'NVDA', 'AMZN', 'META', 'TSLA',
        'JPM', 'V', 'MA', 'JNJ', 'UNH',
        'SPY', 'QQQ', 'TLT', 'GLD',
    ]

    AI_INFRA_SYMBOLS = [
        'CRWD', 'PANW', 'EQIX', 'DLR', 'AMAT', 'LRCX', 'MP', 'LTHM',
        'ROK', 'ABB',
    ]

    def __init__(self, capital: float, risk_profile: str,
                 caio_mode: bool = False):
        self.capital      = capital
        self.risk_profile = risk_profile
        self.risk_aversion = {'conservative': 4.0, 'moderate': 2.5, 'aggressive': 1.0}[risk_profile]
        self.caio_mode    = caio_mode

        self.ALL_SYMBOLS = list(dict.fromkeys(self.CORE_SYMBOLS + self.AI_INFRA_SYMBOLS))

        # Instantiate all modules
        self.data_mgr     = RobustDataManager()
        self.feat_engine  = AdaptiveFeatureEngine()
        self.weight_mgr   = AdaptiveWeightManager()
        self.regime_pred  = RegimeAwarePredictor()
        self.calibrator   = AdaptiveCalibrator(SecureConfig.DB_PATH)
        self.tracker      = PerformanceTracker(SecureConfig.DB_PATH)
        self.stress_ind   = MarketStressIndicator()
        self.decision_eng = IntelligentDecisionEngine(
            self.calibrator, self.stress_ind, self.tracker)
        self.optimizer    = PortfolioOptimizerV2(capital, self.risk_aversion)

        # 4.4 NEW modules
        self.agentic       = AgenticPrototype()
        self.diversifier   = DiversificationModule(capital)
        self.integrity     = IntegrityLayer()
        self.roi_tracker   = ROITracker()
        self.caio          = ChiefAIOfficerMode(enabled=caio_mode)

        # State
        self.market_data:      Dict[str, pd.DataFrame] = {}
        self.predictions:      Dict[str, Dict]          = {}
        self.decisions:        Dict[str, Decision]       = {}
        self.allocation:       Dict[str, float]          = {}
        self.ai_allocation:    Dict[str, float]          = {}
        self.regime_info:      Dict[str, Any]            = {}
        self.macro_data:       Dict[str, pd.Series]      = {}
        self.anomaly_flags:    Dict[str, float]          = {}
        self.integrity_results: Dict[str, Dict]          = {}

        if self.calibrator.should_update():
            self.calibrator.calibrate()

    # ------------------------------------------------------------------
    # MAIN PIPELINE
    # ------------------------------------------------------------------

    def run(self) -> None:
        logger.log(LogLevel.INFO,
                   f"SmartInvest {self.VERSION} ({self.CODENAME}) starting...", "CORE")
        logger.start_timer("full_pipeline")

        # 1. Load data
        self._load_data()
        if not self.market_data:
            logger.log(LogLevel.ERROR, "No market data loaded. Exiting.", "CORE")
            return

        # 2. Macro data
        logger.log(LogLevel.INFO, "Loading macroeconomic data...", "MACRO")
        self.macro_data = self.data_mgr.fetch_macro_data()
        logger.log(LogLevel.SUCCESS, f"Macro series: {list(self.macro_data.keys())}", "MACRO")

        # 3. Agent init (4.4)
        if SecureConfig.AGENTIC_ENABLED:
            self.agentic.execute_task('init_portfolio', {
                'symbols': self.ALL_SYMBOLS,
                'capital': self.capital,
            })

        # 4. Agentic data quality check (4.4)
        dq_result = self.agentic.execute_task('data_quality_check',
                                               {'market_data': self.market_data})
        logger.log(LogLevel.AGENT,
                   f"Data quality pass rate: "
                   f"{dq_result.get('output', {}).get('pass_rate', 0):.1%} "
                   f"(via DataQualityAgent)", "AGENT")

        # 5. Anomaly detection (4.4)
        logger.log(LogLevel.AGENT, "Running anomaly detection agent...", "AGENT")
        self.anomaly_flags = self.agentic.get_anomaly_flags(self.market_data)
        if self.anomaly_flags:
            logger.log(LogLevel.AGENT,
                       f"Anomalies detected: {list(self.anomaly_flags.keys())}", "AGENT")

        # 6. Detect market regime
        self._detect_regime()

        # 7. Train regime-specific models
        self._train_regime_models()

        # 8. Generate predictions + decisions (with agent flags)
        self._generate_predictions()

        # 9. Base portfolio optimization
        self._optimize_portfolio()

        # 10. AI Infra diversification (4.4)
        self._diversify_ai_infra()

        # 11. Portfolio rebalance agent check (4.4)
        if self.ai_allocation:
            rebalance = self.agentic.execute_task('portfolio_rebalance', {
                'current_allocation': self.allocation,
                'target_allocation':  self.ai_allocation,
            })
            if rebalance.get('output', {}).get('rebalance_needed'):
                logger.log(LogLevel.AGENT,
                           f"Rebalance agent: {rebalance['output']['flagged_count']} "
                           "positions need adjustment", "AGENT")

        # 12. Top-down strategy (Chief AI Officer mode)
        strategy_result = self.agentic.execute_task('top_down_strategy', {
            'regime_info': self.regime_info,
            'macro_data':  self.macro_data,
            'decisions':   self.decisions,
            'allocation':  self.ai_allocation or self.allocation,
        })

        # 13. ROI calculation (4.4) — simulated actual returns
        roi_result = self._calculate_roi()

        # 14. Benchmark comparison
        benchmark = self.roi_tracker.compare_to_benchmark(
            self.market_data, self.ai_allocation or self.allocation)

        # 15. Integrity summary
        integrity_summary = self.integrity.get_integrity_summary()

        # 16. Reports
        self._print_main_report()
        self._print_opportunity_cost()
        self.diversifier.print_ai_infra_report(self.ai_allocation, self.market_data)
        self.roi_tracker.print_roi_report(roi_result, benchmark)
        self._print_agent_report()

        # 17. Chief AI Officer briefing (if enabled)
        if self.caio_mode:
            briefing = self.caio.generate_briefing(
                agent_strategy=strategy_result,
                integrity_summary=integrity_summary,
                roi_summary=roi_result,
                regime_info=self.regime_info,
                ai_infra_allocation=self.ai_allocation,
                capital=self.capital,
            )
            print(briefing)

        # 18. Visualize
        if PLOTLY_AVAILABLE:
            self._visualize()

        logger.end_timer("full_pipeline")
        logger.log(LogLevel.SUCCESS, "SmartInvest 4.4 analysis complete.", "CORE")

    # ------------------------------------------------------------------

    def _load_data(self) -> None:
        logger.log(LogLevel.INFO,
                   f"Loading data for {len(self.ALL_SYMBOLS)} symbols...", "DATA")
        logger.start_timer("data_load")

        raw_data = self.data_mgr.fetch_multiple(self.ALL_SYMBOLS, period='3y')

        for sym, df in raw_data.items():
            try:
                # 4.4: AI-ready data preparation
                df_ready = self.data_mgr.prepare_ai_ready_data(df, sym)
                # Feature engineering
                df_feat = self.feat_engine.engineer_features(df_ready)
                if len(df_feat) >= SecureConfig.MIN_HISTORY_DAYS:
                    self.market_data[sym] = df_feat
            except Exception as e:
                logger.log(LogLevel.WARNING, f"Data prep failed for {sym}: {e}", "DATA")

        logger.end_timer("data_load")
        logger.log(LogLevel.SUCCESS,
                   f"Loaded {len(self.market_data)} assets (AI-ready)", "DATA")

    def _detect_regime(self) -> None:
        spy_df = self.market_data.get('SPY')
        self.regime_info = self.regime_pred.detect(spy_df)
        regime = self.regime_info['regime']
        probs  = self.regime_info['probabilities']
        crisis = self.regime_info['crisis_mode']
        prob_str = " | ".join(f"{r.upper()} {p*100:.0f}%"
                              for r, p in probs.items() if p > 0.05)
        logger.log(LogLevel.INFO,
                   f"Regime: {regime.upper()} [{prob_str}]"
                   f"{' ⚠️ CRISIS MODE' if crisis else ''}", "REGIME")

    def _train_regime_models(self) -> None:
        spy_df = self.market_data.get('SPY')
        if spy_df is not None:
            feat_cols = self.feat_engine.get_feature_cols()
            available = [c for c in feat_cols if c in spy_df.columns]
            self.regime_pred.train_regime_models(spy_df, self.regime_info, available)

    def _generate_predictions(self) -> None:
        logger.log(LogLevel.INFO, "Generating predictions + decisions...", "AI")
        feature_cols = self.feat_engine.get_feature_cols()
        regime = self.regime_info.get('regime', 'sideways')

        for symbol, df in self.market_data.items():
            try:
                avail_cols = [c for c in feature_cols if c in df.columns]
                if len(avail_cols) < 10:
                    continue

                # Train ensemble
                ensemble = AdvancedMLEnsemble(self.weight_mgr)
                if not ensemble.train(df, avail_cols):
                    continue

                # Predict
                pred_result  = ensemble.predict(df, avail_cols, symbol, regime)
                regime_blend = self.regime_pred.blended_prediction(
                    df, self.regime_info['probabilities'], avail_cols)
                macro_feat   = self.data_mgr.build_macro_features(
                    self.macro_data, df.index[-1])
                geo          = self.data_mgr.fetch_geo_sentiment(symbol)

                final_pred = pred_result['prediction']
                if regime_blend is not None:
                    final_pred = 0.80 * final_pred + 0.20 * regime_blend

                geo_risk   = geo.get('risk_score', 0.1)
                final_pred = final_pred * (1 - geo_risk * 0.15)

                cal_conf = self.calibrator.calibrate_confidence(
                    pred_result['confidence'], symbol, regime)
                dq = self.data_mgr.get_data_quality(symbol)

                # 4.4: Agent anomaly flag
                anomaly_score = self.anomaly_flags.get(symbol, 0.0)
                agent_flag    = anomaly_score > SecureConfig.AGENT_ANOMALY_THRESHOLD

                # 4.4: Integrity check on individual model predictions
                pred_list = list(pred_result['individual'].values())
                int_result = self.integrity.validate_model(symbol, pred_list)
                self.integrity_results[symbol] = int_result

                # Decision (with 4.4 agent flags)
                decision = self.decision_eng.decide(
                    symbol=symbol,
                    raw_prediction=final_pred,
                    raw_confidence=pred_result['confidence'],
                    individual_preds=pred_result['individual'],
                    weights_used=pred_result['weights_used'],
                    market_data=df,
                    regime_info=self.regime_info,
                    sector=self.feat_engine.get_sector(symbol),
                    data_quality=dq,
                    agent_flag=agent_flag,
                    anomaly_score=anomaly_score,
                )

                # 4.4: Log decision trace
                self.integrity.log_decision_trace(decision, int_result)

                self.decisions[symbol]   = decision
                self.predictions[symbol] = {
                    'prediction':            final_pred,
                    'confidence':            pred_result['confidence'],
                    'calibrated_confidence': decision.calibrated_confidence,
                    'uncertainty':           pred_result['uncertainty'],
                    'geo_risk':              geo_risk,
                    'individual':            pred_result['individual'],
                    'weights_used':          pred_result['weights_used'],
                    'feature_importance':    ensemble.get_feature_importance(),
                    'anomaly_score':         anomaly_score,
                    'agent_flag':            agent_flag,
                    'integrity_ok':          int_result.get('integrity_ok', True),
                }

                self.tracker.record_prediction(
                    symbol=symbol,
                    predicted_return=final_pred,
                    confidence=pred_result['confidence'],
                    calibrated_confidence=decision.calibrated_confidence,
                    model_disagreement=decision.model_disagreement,
                    market_stress=decision.market_stress,
                    decision=decision.decision,
                    would_have_traded=decision.would_have_traded,
                    regime=regime,
                )
                self.tracker.record_decision(decision)

                flag_str = "🤖⚡" if agent_flag else ""
                int_str  = "🛡️✓" if int_result.get('integrity_ok') else "🛡️✗"
                logger.log(LogLevel.SUCCESS,
                           f"{symbol}: {decision.decision} "
                           f"(pred={final_pred*100:.1f}%, conf={decision.calibrated_confidence:.2f}) "
                           f"{flag_str}{int_str}", "DECISION")

            except Exception as e:
                logger.log(LogLevel.WARNING, f"Failed {symbol}: {e}", "AI")

    def _optimize_portfolio(self) -> None:
        logger.log(LogLevel.INFO, "Optimizing base portfolio...", "OPTIMIZER")
        self.allocation = self.optimizer.optimize(
            self.decisions, self.market_data, self.regime_info)
        logger.log(LogLevel.SUCCESS,
                   f"Base allocation: {len(self.allocation)} assets, "
                   f"total={sum(self.allocation.values()):,.0f}", "OPTIMIZER")

    def _diversify_ai_infra(self) -> None:
        """4.4: Add AI infrastructure positions on top of base allocation."""
        logger.log(LogLevel.INFO, "Applying AI infra diversification...", "DIVERSIFY")
        self.ai_allocation = self.diversifier.optimize_ai_infra_allocation(
            self.market_data, self.decisions, self.allocation)

    def _calculate_roi(self) -> Dict[str, Any]:
        """4.4: Calculate ROI using simulated actual returns (30d lookback)."""
        predictions_dict  = {}
        actual_returns    = {}
        symbol_categories: Dict[str, str] = {}

        all_ai_syms = {}
        for cat, syms in SecureConfig.AI_INFRA_INSTRUMENTS.items():
            for sym in syms:
                all_ai_syms[sym] = cat

        for symbol, pred_data in self.predictions.items():
            df = self.market_data.get(symbol)
            if df is None or len(df) < 31:
                continue
            predictions_dict[symbol]  = pred_data['prediction']
            actual_return = float(df['Close'].iloc[-1] / df['Close'].iloc[-31] - 1)
            actual_returns[symbol]    = actual_return
            symbol_categories[symbol] = all_ai_syms.get(
                symbol, self.feat_engine.get_sector(symbol))

        if not predictions_dict:
            return {'productivity_score': 0.0, 'alpha': 0.0,
                    'bubble_count': 0, 'bubble_symbols': [],
                    'n_tracked': 0, 'overall_mse': 0.0,
                    'ai_generating_alpha': False}

        return self.roi_tracker.calculate_ai_roi(
            predictions_dict, actual_returns, symbol_categories)

    # ------------------------------------------------------------------
    # REPORTING
    # ------------------------------------------------------------------

    def _print_main_report(self) -> None:
        W = 100
        print("\n" + "═" * W)
        print(f"  SMARTINVEST {self.VERSION} — {self.CODENAME.upper()}")
        r = self.regime_info
        print(f"  Regime: {r.get('regime','?').upper()} "
              f"{'⚠️ CRISIS' if r.get('crisis_mode') else ''} | "
              f"Vol: {r.get('recent_vol',0)*100:.1f}% | "
              f"Momentum: {r.get('recent_mom',0)*100:+.1f}%")
        print("═" * W)

        buy_d  = {s: d for s, d in self.decisions.items() if d.decision == 'BUY'}
        sell_d = {s: d for s, d in self.decisions.items() if d.decision == 'SELL'}
        hold_d = {s: d for s, d in self.decisions.items() if d.decision == 'HOLD'}

        def print_section(title: str, items: Dict[str, Decision], color: str = ''):
            if not items:
                return
            print(f"\n  {title}")
            print(f"  {'Symbol':<8} {'Pred%':>7} {'CalConf':>8} {'Stress':>8} "
                  f"{'DisAgr':>8} {'DQ':>5} {'Agent':>6} {'Integrity':>10} Reason")
            print("  " + "─" * 90)
            for sym, d in sorted(items.items(), key=lambda x: -x[1].calibrated_confidence):
                pdata = self.predictions.get(sym, {})
                a_flag  = "⚡" if d.agent_flag else "  "
                int_ok  = "✓" if pdata.get('integrity_ok', True) else "✗"
                print(f"  {sym:<8} {d.raw_prediction*100:>6.1f}% "
                      f"{d.calibrated_confidence:>8.2f} "
                      f"{d.market_stress:>8.2f} "
                      f"{d.model_disagreement:>8.4f} "
                      f"{d.data_quality:>5.2f} "
                      f"{a_flag:>6} "
                      f"{int_ok:>10} "
                      f"{d.reason[:35]}")

        print_section("📈 BUY SIGNALS", buy_d)
        print_section("📉 SELL SIGNALS", sell_d)
        print_section("⚪ HOLD SIGNALS", hold_d)

        # Portfolio allocation
        alloc = self.ai_allocation or self.allocation
        if alloc:
            print(f"\n  {'─' * 90}")
            print(f"  💼 PORTFOLIO ALLOCATION (€{sum(alloc.values()):,.0f} / €{self.capital:,.0f})")
            print(f"  {'─' * 90}")
            print(f"  {'Symbol':<10} {'Allocation':>14} {'%Port':>8} {'Sector':<18} {'Category'}")
            print("  " + "─" * 70)

            all_ai_syms = {sym: cat for cat, syms in SecureConfig.AI_INFRA_INSTRUMENTS.items()
                           for sym in syms}

            for sym, amt in sorted(alloc.items(), key=lambda x: -x[1]):
                pct = amt / self.capital * 100
                sector   = self.feat_engine.get_sector(sym)
                category = all_ai_syms.get(sym, 'core')
                print(f"  {sym:<10} €{amt:>12,.0f} {pct:>7.1f}%  {sector:<18} {category}")

            cash = self.capital - sum(alloc.values())
            if cash > 0:
                print(f"  {'CASH':<10} €{cash:>12,.0f} {cash/self.capital*100:>7.1f}%  "
                      f"{'cash':<18} core")

        print("\n" + "═" * W + "\n")

    def _print_opportunity_cost(self) -> None:
        summary = self.tracker.get_summary()
        W = 100
        total_trades = summary['correct'] + summary['wrong']
        print(f"{'─' * W}")
        print(f"  📊 OPPORTUNITY COST TRACKER")
        print(f"{'─' * W}")
        print(f"  ✅ Correct trades:        {summary['correct']:3d}  "
              f"(+{summary['correct'] * SecureConfig.OUTCOME_VALUES['correct_trade']:.0f} pts)")
        print(f"  ❌ Wrong trades:          {summary['wrong']:3d}  "
              f"({summary['wrong'] * SecureConfig.OUTCOME_VALUES['wrong_trade']:.0f} pts)")
        print(f"  ⚠️  Missed opportunities: {summary['missed']:3d}  "
              f"({summary['missed'] * SecureConfig.OUTCOME_VALUES['missed_opportunity']:.0f} pts)")
        print(f"  ⚪ Neutral HOLDs:         {summary['neutral']:3d}  (0 pts)")
        print(f"\n  📊 Total Value: {summary['total_value']:.1f} pts")
        if total_trades > 0:
            win_rate = summary['correct'] / total_trades
            print(f"  🎯 Win Rate: {win_rate:.1%}")
        print("─" * W + "\n")

    def _print_agent_report(self) -> None:
        W = 100
        print(f"{'─' * W}")
        print(f"  🤖 AGENTIC PROTOTYPE REPORT")
        print(f"{'─' * W}")

        history = self.agentic.get_task_history()
        print(f"  Agent Tasks Executed: {len(history)}")
        for t in history[:8]:
            status = '✅' if not t.get('error') else '❌'
            print(f"  {status} {t['task_type']:<30} [{t.get('executed_at','?')[:19]}]")

        if self.anomaly_flags:
            print(f"\n  ⚡ Anomaly Flags ({len(self.anomaly_flags)}):")
            for sym, score in sorted(self.anomaly_flags.items(), key=lambda x: -x[1])[:5]:
                print(f"     {sym}: score={score:.2f}")
        else:
            print(f"\n  ✅ No anomalies detected by agents")

        # Integrity summary
        int_summary = self.integrity.get_integrity_summary()
        print(f"\n  🛡️  Integrity:")
        print(f"     Traces logged:    {int_summary.get('total_traced', 0)}")
        print(f"     Integrity rate:   {int_summary.get('integrity_rate', 1.0):.1%}")
        if int_summary.get('poisoning_suspected', 0) > 0:
            print(f"     ⚠️  Poisoning suspected: {int_summary['poisoning_suspected']} case(s)")

        print("─" * W + "\n")

    def _visualize(self) -> None:
        """Extended Plotly dashboard for 4.4 (adds AI Factories + Agent panels)."""
        if not PLOTLY_AVAILABLE or not self.decisions:
            return

        alloc = self.ai_allocation or self.allocation

        fig = make_subplots(
            rows=3, cols=3,
            subplot_titles=(
                'Decision Distribution',
                'Raw vs Calibrated Confidence',
                'Portfolio Allocation',
                'Regime Probabilities',
                'AI Infra Category Allocation',
                'Model Predictions Spread',
                'Anomaly Scores',
                'Integrity Status',
                'ROI: Predicted vs Actual',
            ),
            specs=[
                [{'type': 'domain'}, {'type': 'scatter'}, {'type': 'bar'}],
                [{'type': 'bar'}, {'type': 'bar'}, {'type': 'box'}],
                [{'type': 'bar'}, {'type': 'bar'}, {'type': 'scatter'}],
            ]
        )

        # 1. Decision pie
        counts = {'BUY': 0, 'SELL': 0, 'HOLD': 0}
        for d in self.decisions.values():
            counts[d.decision] = counts.get(d.decision, 0) + 1
        fig.add_trace(go.Pie(
            labels=list(counts.keys()), values=list(counts.values()),
            marker=dict(colors=['#2ecc71', '#e74c3c', '#95a5a6'])
        ), row=1, col=1)

        # 2. Raw vs calibrated confidence
        syms    = list(self.decisions.keys())
        raw_c   = [self.decisions[s].raw_confidence for s in syms]
        cal_c   = [self.decisions[s].calibrated_confidence for s in syms]
        colors  = ['#2ecc71' if self.decisions[s].decision == 'BUY' else
                   '#e74c3c' if self.decisions[s].decision == 'SELL' else '#95a5a6'
                   for s in syms]
        # Agent flagged = different marker
        marker_syms = ['star' if self.decisions[s].agent_flag else 'circle' for s in syms]
        fig.add_trace(go.Scatter(
            x=raw_c, y=cal_c, mode='markers+text',
            text=syms, textposition='top center',
            marker=dict(size=9, color=colors, symbol=marker_syms),
            showlegend=False
        ), row=1, col=2)
        fig.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1], mode='lines',
            line=dict(color='gray', dash='dash'), showlegend=False
        ), row=1, col=2)

        # 3. Portfolio allocation
        if alloc:
            alloc_syms = list(alloc.keys())
            alloc_vals = [alloc[s] / self.capital * 100 for s in alloc_syms]
            all_ai_syms = {sym: cat for cat, syms in SecureConfig.AI_INFRA_INSTRUMENTS.items()
                           for sym in syms}
            bar_colors = ['#e67e22' if s in all_ai_syms else '#3498db' for s in alloc_syms]
            fig.add_trace(go.Bar(
                x=alloc_syms, y=alloc_vals,
                marker_color=bar_colors, name='Allocation %'
            ), row=1, col=3)

        # 4. Regime probabilities
        probs = self.regime_info.get('probabilities', {})
        if probs:
            reg_colors = {'bull': '#2ecc71', 'bear': '#e74c3c',
                          'sideways': '#f39c12', 'crisis': '#8e44ad'}
            fig.add_trace(go.Bar(
                x=list(probs.keys()),
                y=[p * 100 for p in probs.values()],
                marker_color=[reg_colors.get(r, '#95a5a6') for r in probs.keys()],
                name='Regime %'
            ), row=2, col=1)

        # 5. AI Infra Category Allocation (4.4 NEW)
        all_ai_syms = {sym: cat for cat, syms in SecureConfig.AI_INFRA_INSTRUMENTS.items()
                       for sym in syms}
        cat_totals: Dict[str, float] = {}
        for sym, amt in alloc.items():
            cat = all_ai_syms.get(sym)
            if cat:
                cat_totals[cat] = cat_totals.get(cat, 0) + amt
        if cat_totals:
            fig.add_trace(go.Bar(
                x=list(cat_totals.keys()),
                y=[v / self.capital * 100 for v in cat_totals.values()],
                marker_color='#e67e22', name='AI Infra %'
            ), row=2, col=2)

        # 6. Model prediction spread
        model_preds: Dict[str, List[float]] = {}
        for sym, pdata in self.predictions.items():
            for model, val in pdata.get('individual', {}).items():
                model_preds.setdefault(model, []).append(val * 100)
        for model, vals in model_preds.items():
            fig.add_trace(go.Box(y=vals, name=model.upper(), showlegend=False),
                          row=2, col=3)

        # 7. Anomaly scores (4.4 NEW)
        if self.anomaly_flags:
            a_syms = list(self.anomaly_flags.keys())
            a_vals = [self.anomaly_flags[s] for s in a_syms]
            a_colors = ['#e74c3c' if v > SecureConfig.AGENT_ANOMALY_THRESHOLD
                        else '#f39c12' for v in a_vals]
            fig.add_trace(go.Bar(
                x=a_syms, y=a_vals,
                marker_color=a_colors, name='Anomaly Score'
            ), row=3, col=1)

        # 8. Integrity status (4.4 NEW)
        int_data = [(sym, 1 if d.get('integrity_ok', True) else 0)
                    for sym, d in self.integrity_results.items()]
        if int_data:
            i_syms, i_vals = zip(*int_data)
            i_colors = ['#2ecc71' if v == 1 else '#e74c3c' for v in i_vals]
            fig.add_trace(go.Bar(
                x=i_syms, y=i_vals,
                marker_color=i_colors, name='Integrity OK'
            ), row=3, col=2)

        # 9. ROI: Predicted vs Actual (4.4 NEW)
        roi_syms, roi_preds, roi_actuals = [], [], []
        for sym, pdata in self.predictions.items():
            df = self.market_data.get(sym)
            if df is not None and len(df) >= 31:
                actual = float(df['Close'].iloc[-1] / df['Close'].iloc[-31] - 1)
                roi_syms.append(sym)
                roi_preds.append(pdata['prediction'] * 100)
                roi_actuals.append(actual * 100)

        if roi_syms:
            fig.add_trace(go.Scatter(
                x=roi_preds, y=roi_actuals, mode='markers+text',
                text=roi_syms[:12], textposition='top center',
                marker=dict(size=8, color='#9b59b6'),
                name='Pred vs Actual', showlegend=False
            ), row=3, col=3)
            lim = max(abs(max(roi_preds + roi_actuals, default=5)),
                      abs(min(roi_preds + roi_actuals, default=-5)))
            fig.add_trace(go.Scatter(
                x=[-lim, lim], y=[-lim, lim], mode='lines',
                line=dict(color='gray', dash='dash'), showlegend=False
            ), row=3, col=3)

        fig.update_layout(
            title_text=(f"SmartInvest 4.4 — {self.regime_info.get('regime','?').upper()} "
                        f"| Agentic Dashboard"),
            template='plotly_dark',
            height=1100,
            showlegend=False,
        )
        fig.show()


# =============================================================================
# MAIN ENTRY POINT
# =============================================================================

def main():
    W = 100
    print("\n" + "█" * W)
    print(f"   🧠 SMARTINVEST 4.4 — AGENTIC PROTOTYPE EDITION")
    print(f"   Autonomous AI Agents + AI Infrastructure Diversification")
    print("█" * W)
    print()
    print("   What's new vs 4.3:")
    print("   🤖 AgenticPrototype — anomaly detection, portfolio rebalancing, data quality agents")
    print("   📊 AIReadyDataModule — FinBERT sentiment + 5-sigma outlier cleaning")
    print("   🏗️  DiversificationModule — cybersecurity, data centers, optics, robotics, rare earth")
    print("   🛡️  IntegrityLayer — model drift, poisoning detection, decision traces")
    print("   💰 ROITracker — productivity score, alpha tracking, bubble warnings")
    print("   🏛️  ChiefAIOfficerMode — enterprise AI strategy briefing (optional)")
    print("   📈 Extended Plotly dashboard — AI Factories + Agent + ROI panels")
    print()
    print("─" * W)
    print("⚠️  LEGAL DISCLAIMER:")
    print("    This is EDUCATIONAL SOFTWARE — NOT financial advice.")
    print("    You CAN and MAY LOSE your entire investment.")
    print("    Past performance does NOT guarantee future results.")
    print("    Always consult a licensed financial advisor.")
    print("─" * W)

    response = input("\n    Type 'I ACCEPT' to continue (or press Enter to exit): ").strip().upper()
    if response != 'I ACCEPT':
        print("\n❌ Exiting. Goodbye.\n")
        sys.exit(0)

    print("\n✅ Disclaimer accepted. Proceeding...\n")

    # --- Capital ---
    while True:
        try:
            capital_str = input("💰 Enter capital amount (EUR): ").replace(',', '').strip()
            capital = float(capital_str)
            if capital > 0:
                break
            print("   ⚠️  Please enter a positive amount.")
        except ValueError:
            print("   ⚠️  Invalid input. Please enter a number.")

    # --- Risk profile ---
    print("\n📊 Select risk profile:")
    print("   1️⃣  Conservative  — bonds heavy, low volatility")
    print("   2️⃣  Moderate      — balanced growth and safety")
    print("   3️⃣  Aggressive    — maximum return, high risk")
    while True:
        choice = input("\nYour choice (1/2/3): ").strip()
        if choice in ('1', '2', '3'):
            risk_map = {'1': 'conservative', '2': 'moderate', '3': 'aggressive'}
            risk_profile = risk_map[choice]
            break
        print("   ⚠️  Please enter 1, 2, or 3.")

    # --- Chief AI Officer Mode ---
    print("\n🏛️  Enable Chief AI Officer Mode? (Enterprise strategy briefing)")
    caio_choice = input("   Type 'YES' to enable (or Enter to skip): ").strip().upper()
    caio_mode = caio_choice == 'YES'

    # --- Config warnings ---
    cfg_warnings = SecureConfig.validate()
    if cfg_warnings:
        print()
        for w in cfg_warnings:
            print(f"⚠️  {w}")
        print()

    caio_label = " + CAIO MODE" if caio_mode else ""
    print(f"\n🚀 Initializing SmartInvest 4.4 for €{capital:,.2f} "
          f"[{risk_profile}{caio_label}]...\n")

    try:
        system = SmartInvest44(capital, risk_profile, caio_mode=caio_mode)
        system.run()

        print(f"\n✅ SmartInvest 4.4 analysis complete!")
        print(f"📁 Logs:      {SecureConfig.LOGS_DIR}")
        print(f"📁 Decisions: {SecureConfig.DECISIONS_DIR}")
        print(f"📁 Traces:    {SecureConfig.TRACES_DIR}")
        print(f"📁 Database:  {SecureConfig.DB_PATH}")
        print(f"\n🙏 Thank you for using SmartInvest 4.4 — Agentic Prototype Edition!\n")

    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user.\n")
    except Exception as e:
        logger.log(LogLevel.CRITICAL, f"Fatal error: {e}", "MAIN")
        traceback.print_exc()


if __name__ == "__main__":
    main()